# 확장 실험 — 1 fold 스크리닝

## 확정된 것 (건드리지 않는다)

```
GIN[base]   봉인 최종 macro F1  0.8662     <- 논문 수치. 고정.
            개발 구간           0.8445 ± 0.0148
```

이 노트북의 어떤 결과도 위 숫자를 바꾸지 않는다.

## 왜 1 fold 인가

셀 19에서 나온 fold 별 차분의 표준편차가 실측 **약 0.02** 다.

| 실제 효과 | 1 fold 에서 부호를 틀릴 확률 |
|---:|---:|
| 0.005 | 40% (동전 던지기) |
| 0.02 | 16% |
| **0.04** | **2%** |
| 0.06 | 0.1% |

**0.04 보다 큰 효과는 1 fold 로 판별된다.** 이전 실험에서 5-fold 가 꼭 필요했던 건
네 설정이 서로 0.005~0.026 차이라 전부 위험 구간에 있었기 때문이다.

지금은 아직 어떤 방향이 살아있는지 모르는 탐색 국면이다. 여기서 5배 비용을 치르면
시도할 수 있는 아이디어 수가 5분의 1이 된다. 정밀도는 후보가 좁혀진 뒤에 산다.

## 순서

| 단계 | 내용 | 프로토콜 | 비용 |
|---|---|---|---|
| ① | 극좌표 CNN 단독 + **오라클 진단** | 1 fold | 25분 |
| ② | 클래스 가중치 완화 `alpha=0.5` | 1 fold | 20분 |
| ③ | 게이트 융합 (①통과 시에만) | 1 fold | 35분 |
| ④ | 살아남은 후보 | 5 fold | 1.5h |

**첫 45분 안에 융합을 할지 말지, 가중치 완화가 유망한지 둘 다 알 수 있다.**

## 규칙 — 지금 정하고 지킨다

> 모든 개발·비교·선택은 **개발 구간**에서만 한다.
> 봉인 집합은 최종 후보 1개당 **한 번씩만** 채점하고, 그 결과를 보고 아무것도 바꾸지 않는다.
> **결과가 나쁘게 나와도 그대로 보고한다.**

## 반드시 GPU 런타임으로


# 1. 환경 준비

기존 분할 파일과 캐시를 **그대로 재사용**한다. 분할을 다시 만들면 이전 결과와
비교가 성립하지 않는다.


In [ ]:
!pip install -q torch_geometric

import os, sys, time, math, json, random, hashlib, warnings, collections
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("[경고] GPU 런타임이 아니다.")

from google.colab import drive
drive.mount('/content/drive')

DRIVE      = '/content/drive/MyDrive/ACK2026_Wafer'
RAW_PKL    = f'{DRIVE}/MIR-WM811K/MIR-WM811K/Python/WM811K_defects.pkl'
SPEC_DIR   = f'{DRIVE}/Train_Model/spec_v1'
CACHE_DIR  = f'{SPEC_DIR}/results'          # 기존 캐시 재사용 (GIN[base] 가 여기 있다)
CKPT_DIR   = f'{SPEC_DIR}/checkpoints'
SPLIT_PATH = f'{SPEC_DIR}/split_sealed_grouped_seed42.npz'

# 그래프 전용(기존)과 극좌표 포함(신규)을 **별도 파일**로 둔다.
# 극좌표는 627MB 라 그래프만 쓸 때 끌고 다닐 이유가 없다.
GRAPH_NPZ  = '/content/wafer_spec.npz'
POLAR_NPZ  = '/content/wafer_spec_polar.npz'
DRIVE_GRAPH= f'{DRIVE}/Train_Model/Training_Data/wafer_spec.npz'
DRIVE_POLAR= f'{DRIVE}/Train_Model/Training_Data/wafer_spec_polar.npz'

assert os.path.exists(SPLIT_PATH), (
    f"분할 파일이 없다: {SPLIT_PATH}\n"
    "먼저 ACK2026_Wafer_Spec_Colab.ipynb 를 실행해 분할을 만들어야 한다.")
print("분할 파일 확인:", SPLIT_PATH)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 17.8 MB/s eta 0:00:00
torch 2.11.0+cu128 | CUDA: True
Mounted at /content/drive
분할 파일 확인: /content/drive/MyDrive/ACK2026_Wafer/Train_Model/spec_v1/split_sealed_grouped_seed42.npz


# 2. 전처리 함수


In [ ]:
"""WM-811K 웨이퍼 맵 전처리 v2.

v1(`Data_Preprocessing/Wafer Data Preprocessing6.ipynb`) 대비 변경점

1. 노이즈 필터를 '다이 피치' 단위로 재정의하고, 삭제 대신 플래그로 보존
   - v1: eps=0.10 (정규화 좌표) -> 웨이퍼 크기에 따라 실제 반경이 1.3~8.6 다이로 요동.
     Center 60%, Scratch 59%, Loc 51% 의 결함 다이가 삭제되고 있었음.
   - v2: DBSCAN 을 다이(die) 단위 좌표에서 수행하여 eps 가 항상 동일한 물리적 거리.
     노이즈로 판정된 점도 지우지 않고 `is_noise` 특징/채널로 남김.

2. 유효 다이(waferMap > 0) 기준 반경 정규화
   - v1: rmax = max(w, h) / 2 로 x, y 를 동시에 나눔 -> 종횡비가 다른 웨이퍼(전체의 66%)
     에서 원이 타원으로 왜곡. 최외곽 반경이 웨이퍼마다 0.955~1.037 로 흔들림.
   - v2: 중심 = 유효 다이 무게중심, R = 유효 다이의 최대 반경.
     정의상 모든 결함 다이가 r <= 1 이 되어 클리핑 손실이 0.

3. 극좌표 이미지를 3채널 밀도 맵으로 교체
   - ch0 결함 밀도 = (빈 내 결함 다이 수) / (빈 내 유효 다이 수)
   - ch1 유효 다이 마스크 = 그 위치에 웨이퍼가 존재하는가
   - ch2 노이즈 판정 결함 밀도
   - v1 은 1채널 이진 대입(`= 1.0`)이라 Edge-Ring 결함의 19% 가 같은 픽셀에 겹쳐 소실됐고,
     "결함 없음"과 "다이 없음"을 구분할 수 없었음.

5. GCN 입력 강화
   - 반경 기반 그래프 대신 k-NN 그래프(k 고정) -> 평균 차수가 0.38~73.8 로 벌어지던 문제 해소.
   - 노드 특징을 (x, y) 2차원에서 11차원으로 확장.

주의: 전처리는 라벨(failureType)을 일절 사용하지 않는다. 패턴별로 파라미터를 다르게 주면
추론 시점에 재현할 수 없는 라벨 누수가 된다.
"""


import numpy as np
from scipy.spatial import cKDTree
from sklearn.cluster import DBSCAN

# 8대 불량 패턴 라벨 인코딩. 전처리·데이터셋·학습이 공유하므로 여기 한 곳에만 둔다.
LABEL_MAP = {
    "Center": 0, "Donut": 1, "Edge-Loc": 2, "Edge-Ring": 3,
    "Loc": 4, "Random": 5, "Scratch": 6, "Near-full": 7,
}
REVERSE_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

# 노드 특징 벡터의 열 인덱스. 증강(회전/반전)이 이 순서에 의존하므로 변경 시 주의.
FEAT_X, FEAT_Y = 0, 1          # 회전/반전 대상
FEAT_R = 2
FEAT_COS, FEAT_SIN = 3, 4      # 회전/반전 대상
FEAT_DENS2 = 5
FEAT_DENS4 = 6
FEAT_KNN_DIST = 7
FEAT_LOCAL_RATIO = 8
FEAT_IS_NOISE = 9
FEAT_LOG_N = 10
NUM_NODE_FEATURES = 11

# 그래프 수준 스칼라 (분류기에 직접 주입)
NUM_GRAPH_SCALARS = 4

POLAR_CHANNELS = 3


class WaferPreprocessorV2:
    """웨이퍼 맵 한 장을 (그래프, 극좌표 이미지, 스칼라) 3종 표현으로 변환한다.

    Parameters
    ----------
    r_bins, theta_bins : 극좌표 이미지 해상도.
    knn_k : 각 결함 노드가 연결할 최근접 이웃 수. 방향성 엣지로 저장하고 학습 시 대칭화한다.
    noise_eps_dies : DBSCAN 반경. **다이 개수 단위**라 웨이퍼 크기와 무관하게 일정하다.
    noise_min_samples : DBSCAN 최소 샘플 수.
    drop_noise : True 면 노이즈 판정 결함을 실제로 삭제한다. 기본 False(플래그만 부여).
        Random 패턴은 산발 노이즈 자체가 클래스 정의이고 Scratch 는 폭 1~2 다이의 선이라
        삭제하면 신호가 사라진다. 삭제 대신 모델이 판단하도록 정보만 넘긴다.
    density_radii : 국소 밀도 특징을 계산할 반경(다이 단위).
    """

    def __init__(
        self,
        r_bins: int = 64,
        theta_bins: int = 64,
        knn_k: int = 8,
        noise_eps_dies: float = 2.0,
        noise_min_samples: int = 3,
        drop_noise: bool = False,
        density_radii: tuple[float, float] = (2.0, 4.0),
        include_polar: bool = True,
    ):
        self.r_bins = r_bins
        self.theta_bins = theta_bins
        self.knn_k = knn_k
        self.noise_eps_dies = noise_eps_dies
        self.noise_min_samples = noise_min_samples
        self.drop_noise = drop_noise
        self.density_radii = density_radii
        # 순수 graph 모델(GCN/GraphSAGE/GAT ...)만 비교할 때는 극좌표 이미지가 필요 없다.
        # 끄면 저장 용량과 RAM 이 크게 줄고 전처리도 빨라진다.
        # theta_bins 는 이미지가 없어도 증강 회전 단위로 계속 쓰이므로 그대로 둔다.
        self.include_polar = include_polar

    # ------------------------------------------------------------------
    # [개선 2] 웨이퍼 기하 구조 추정
    # ------------------------------------------------------------------
    @staticmethod
    def wafer_geometry(wafer_map: np.ndarray):
        """유효 다이만으로 웨이퍼 원의 중심과 반경을 추정한다.

        배열 중심 `(w/2, h/2)` 과 `max(w, h)/2` 를 쓰던 v1 과 달리 종횡비, 여백,
        다이 크기에 영향받지 않는다. 반환하는 R 은 유효 다이의 최대 반경이므로
        모든 유효 다이(=결함 다이 포함)가 r <= 1 을 만족한다.
        """
        vy, vx = np.nonzero(wafer_map > 0)
        if len(vx) == 0:
            return None
        cx, cy = vx.mean(), vy.mean()
        dv = np.hypot(vx - cx, vy - cy)
        radius = float(dv.max())
        if radius <= 0:
            return None
        return vx, vy, cx, cy, radius

    # ------------------------------------------------------------------
    def process_wafer(self, wafer_map: np.ndarray) -> dict | None:
        wafer_map = np.asarray(wafer_map)
        geom = self.wafer_geometry(wafer_map)
        if geom is None:
            return None
        vx, vy, cx, cy, radius = geom

        by, bx = np.nonzero(wafer_map == 2)
        if len(bx) == 0:
            return None

        # 다이 단위 좌표(1 = 다이 1개 간격). 거리 기반 파라미터는 전부 이 좌표계에서 정의한다.
        defect_px = np.column_stack((bx - cx, cy - by)).astype(np.float64)
        die_px = np.column_stack((vx - cx, cy - vy)).astype(np.float64)

        # [개선 1] 다이 단위 DBSCAN -> 웨이퍼 크기와 무관하게 동일한 물리적 필터
        if len(defect_px) >= self.noise_min_samples:
            labels = DBSCAN(
                eps=self.noise_eps_dies, min_samples=self.noise_min_samples
            ).fit(defect_px).labels_
            is_noise = (labels == -1)
        else:
            is_noise = np.ones(len(defect_px), dtype=bool)

        if self.drop_noise and (~is_noise).sum() >= 3:
            defect_px = defect_px[~is_noise]
            is_noise = is_noise[~is_noise]

        node_feat = self._node_features(defect_px, die_px, radius, is_noise)
        edge_index = self._knn_edges(defect_px)
        scalars = np.array(
            [
                np.log1p(len(defect_px)) / 10.0,
                len(defect_px) / len(die_px),
                np.log1p(len(die_px)) / 10.0,
                radius / 100.0,
            ],
            dtype=np.float32,
        )

        result = {
            "node_feat": node_feat.astype(np.float16),
            "edge_index": edge_index.astype(np.int16),  # 노드 수 < 32767 보장
            "scalars": scalars,
            "n_defect": int(len(defect_px)),
            "n_die": int(len(die_px)),
            "radius": float(radius),
            "noise_ratio": float(is_noise.mean()),
        }
        if self.include_polar:
            result["polar"] = self._polar_density(
                defect_px, die_px, radius, is_noise).astype(np.float16)
        return result

    # ------------------------------------------------------------------
    # [개선 5] 노드 특징 11차원
    # ------------------------------------------------------------------
    def _node_features(self, defect_px, die_px, radius, is_noise) -> np.ndarray:
        n = len(defect_px)
        x = defect_px[:, 0] / radius
        y = defect_px[:, 1] / radius
        r = np.hypot(x, y)
        theta = np.arctan2(y, x)

        defect_tree = cKDTree(defect_px)
        die_tree = cKDTree(die_px)

        r_small, r_large = self.density_radii
        # 자기 자신이 항상 포함되므로 -1
        dens_small = np.array(defect_tree.query_ball_point(defect_px, r_small, return_length=True)) - 1.0
        dens_large = np.array(defect_tree.query_ball_point(defect_px, r_large, return_length=True)) - 1.0
        die_large = np.array(die_tree.query_ball_point(defect_px, r_large, return_length=True))

        # k-NN 평균 거리(다이 단위). 노드가 k+1개 미만이면 있는 만큼만 사용한다.
        k = min(self.knn_k + 1, n)
        if k >= 2:
            dists, _ = defect_tree.query(defect_px, k=k)
            knn_mean = dists[:, 1:].mean(axis=1)
        else:
            knn_mean = np.full(n, 10.0)

        feat = np.zeros((n, NUM_NODE_FEATURES), dtype=np.float32)
        feat[:, FEAT_X] = x
        feat[:, FEAT_Y] = y
        feat[:, FEAT_R] = r
        feat[:, FEAT_COS] = np.cos(theta)
        feat[:, FEAT_SIN] = np.sin(theta)
        feat[:, FEAT_DENS2] = np.log1p(np.clip(dens_small, 0, None))
        feat[:, FEAT_DENS4] = np.log1p(np.clip(dens_large, 0, None))
        feat[:, FEAT_KNN_DIST] = np.clip(knn_mean, 0, 20.0) / 20.0
        feat[:, FEAT_LOCAL_RATIO] = np.clip(dens_large, 0, None) / np.maximum(die_large, 1.0)
        feat[:, FEAT_IS_NOISE] = is_noise.astype(np.float32)
        feat[:, FEAT_LOG_N] = np.log1p(n) / 10.0
        return feat

    # ------------------------------------------------------------------
    # [개선 5] 고정 반경 대신 k-NN 그래프
    # ------------------------------------------------------------------
    def _knn_edges(self, defect_px) -> np.ndarray:
        n = len(defect_px)
        k = min(self.knn_k + 1, n)
        if k < 2:
            return np.zeros((2, 0), dtype=np.int64)
        _, idx = cKDTree(defect_px).query(defect_px, k=k)
        src = np.repeat(np.arange(n), k - 1)
        dst = idx[:, 1:].reshape(-1)
        return np.vstack((src, dst))

    # ------------------------------------------------------------------
    # [개선 3] 3채널 극좌표 밀도 맵
    # ------------------------------------------------------------------
    def _bin_indices(self, pts, radius):
        r = np.hypot(pts[:, 0], pts[:, 1]) / radius
        theta = np.arctan2(pts[:, 1], pts[:, 0])
        ri = np.clip((r * self.r_bins).astype(np.int64), 0, self.r_bins - 1)
        # v1 은 (theta_bins - 1) 을 곱해 마지막 빈이 theta == +pi 한 점만 받았다.
        # theta_bins 를 곱하고 modulo 를 취해야 각 빈의 각도 폭이 균일해진다.
        u = (theta + np.pi) / (2 * np.pi) * self.theta_bins
        # +0.5 후 내림 = 반올림. 빈 k 의 '중심'이 theta = -pi + k*2pi/T 에 놓인다.
        # 내림만 쓰면 빈 경계가 theta = 0, +-pi/2, +-pi 위에 정확히 얹히는데,
        # 격자 좌표의 결함은 이 축 위에 대량으로 존재해서 좌우 반전 시
        # 그래프 트랙과 이미지 트랙이 반 칸 어긋난다(축 위 점은 반전의 고정점이어야 함).
        ti = np.floor(u + 0.5).astype(np.int64) % self.theta_bins
        return ri * self.theta_bins + ti

    def _polar_density(self, defect_px, die_px, radius, is_noise) -> np.ndarray:
        size = self.r_bins * self.theta_bins
        die_count = np.bincount(self._bin_indices(die_px, radius), minlength=size)
        defect_bins = self._bin_indices(defect_px, radius)
        defect_count = np.bincount(defect_bins, minlength=size)
        noise_count = np.bincount(defect_bins[is_noise], minlength=size)

        polar = np.zeros((POLAR_CHANNELS, size), dtype=np.float32)
        denom = np.maximum(die_count, 1)
        polar[0] = defect_count / denom          # 결함 밀도
        polar[1] = (die_count > 0).astype(np.float32)  # 유효 다이 마스크
        polar[2] = noise_count / denom           # 노이즈 판정 결함 밀도
        return polar.reshape(POLAR_CHANNELS, self.r_bins, self.theta_bins)


# 3. 데이터셋 생성 함수


In [ ]:
"""WM811K_defects.pkl -> 학습용 packed .npz 생성 스크립트.

사용 예)
    python src/build_dataset.py \
        --input  /content/drive/MyDrive/ACK2026_Wafer/MIR-WM811K/MIR-WM811K/Python/WM811K_defects.pkl \
        --output /content/drive/MyDrive/ACK2026_Wafer/Train_Model/Training_Data/wafer_v2.npz

가변 길이인 노드/엣지를 하나의 큰 배열 + 오프셋(ptr)으로 묶어 저장한다.
웨이퍼마다 개별 dict 를 피클하던 v1 방식보다 로딩이 빠르고 RAM 을 적게 쓴다.
"""


import argparse
import hashlib
import time

import numpy as np
import pandas as pd



def flatten_label(value):
    """MATLAB 유래의 중첩 배열(['Edge-Ring'] 등)을 문자열로 편다."""
    while isinstance(value, (list, np.ndarray)) and len(value) > 0:
        value = value[0]
    return value




# 사양서 §1.3 배제 사유 코드. 순서대로 판정하고 사유별 개수를 집계한다.
DROP_CODES = ("no_label", "no_valid_die", "no_defect", "too_small", "degenerate")


def wafer_hash(wafer_map):
    """§1.4 중복 판정 키 = (맵 형상, 결함 다이 좌표 집합) 의 해시.

    결함 좌표만으로는 부족하다. 형상이 다른 두 웨이퍼가 같은 좌표 집합을 가질 수 있다.
    """
    m = np.asarray(wafer_map)
    ys, xs = np.nonzero(m == 2)
    h = hashlib.sha1()
    h.update(np.asarray(m.shape, dtype=np.int64).tobytes())
    h.update(np.ascontiguousarray(np.stack([ys, xs]).astype(np.int32)).tobytes())
    return h.hexdigest()


def file_hash(path, chunk=1 << 20):
    """§2.8 산출물 해시. 캐시 경로에 넣어 구 캐시 재사용을 구조적으로 막는다."""
    h = hashlib.sha1()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()[:12]


def build_dataset(input_pkl, output_npz, r_bins=64, theta_bins=64, knn_k=8,
                  noise_eps_dies=2.0, noise_min_samples=3, drop_noise=False, limit=0,
                  include_polar=True, min_valid_die=0, min_defect_die=0):
    """원본 pkl 을 읽어 packed .npz 로 저장한다. 노트북에서도 그대로 호출할 수 있다.

    include_polar=False 면 극좌표 이미지를 만들지도 저장하지도 않는다.
    순수 graph 모델만 비교할 때 쓰면 파일이 훨씬 작아지고 전처리도 빨라진다.

    min_valid_die : §1.3 `too_small` 임계값(§8 미확정 항목). 실측상 유효 다이
        최소가 439 라 300 이하 어떤 값을 줘도 배제 0장이다.
    min_defect_die : 결함 다이(=그래프 노드) 하한. k-NN 이 성립하려면 노드가
        k+1 개 이상이어야 한다. 실측상 9 미만이 33장(전부 Loc)이다.

    §1.4 dup_ids 는 **이 함수 안에서** 산출한다(§2.8). 원본 데이터프레임 기준으로
    바깥에서 계산하면 배제된 웨이퍼 때문에 인덱스가 밀려 무관한 웨이퍼가 같은
    그룹으로 묶인다.
    """
    df = pd.read_pickle(input_pkl)
    if limit:
        df = df.iloc[:limit]
    df = df.reset_index(drop=True)  # v1 은 인덱스를 리셋하지 않아 진행률 출력이 깨졌다
    labels_str = df["failureType"].map(flatten_label)
    lot_codes = pd.factorize(df["lotName"].map(flatten_label).astype(str))[0]

    pp = WaferPreprocessorV2(
        r_bins=r_bins, theta_bins=theta_bins, knn_k=knn_k,
        noise_eps_dies=noise_eps_dies, noise_min_samples=noise_min_samples,
        drop_noise=drop_noise, include_polar=include_polar,
    )

    feats, edges, polars, scalars, labels, lots, hashes = [], [], [], [], [], [], []
    drop_log = {c: 0 for c in DROP_CODES}
    t0 = time.time()
    for i, wafer_map in enumerate(df["waferMap"]):
        label = labels_str.iloc[i]
        if label not in LABEL_MAP:
            drop_log["no_label"] += 1
            continue
        code = _drop_reason(wafer_map, min_valid_die, min_defect_die)
        if code is not None:
            drop_log[code] += 1
            continue
        out = pp.process_wafer(wafer_map)
        if out is None:
            drop_log["degenerate"] += 1
            continue
        feats.append(out["node_feat"])
        edges.append(out["edge_index"])
        if include_polar:
            polars.append(out["polar"])
        scalars.append(out["scalars"])
        labels.append(LABEL_MAP[label])
        lots.append(lot_codes[i])
        hashes.append(wafer_hash(wafer_map))
        if (i + 1) % 2000 == 0:
            print(f"  {i + 1}/{len(df)}  ({time.time() - t0:.0f}s)", flush=True)

    # §1.4 dup_ids — 살아남은 웨이퍼만으로 그룹을 매긴다.
    dup_ids = pd.factorize(pd.Series(hashes))[0].astype(np.int64)

    node_ptr = np.concatenate([[0], np.cumsum([len(f) for f in feats])]).astype(np.int64)
    edge_ptr = np.concatenate([[0], np.cumsum([e.shape[1] for e in edges])]).astype(np.int64)

    edge_index = np.concatenate(edges, axis=1) if edges else np.zeros((2, 0), np.int16)
    # §2.8 int16 오버플로 방어. 엣지는 웨이퍼별 지역 인덱스라 지금은 최대 5,493
    # (한계의 1/6) 이지만, 전처리가 바뀌어 노드가 늘면 조용히 음수로 무너진다.
    if edge_index.dtype == np.int16:
        max_nodes = int(np.diff(node_ptr).max()) if len(node_ptr) > 1 else 0
        assert max_nodes <= np.iinfo(np.int16).max, (
            f"웨이퍼당 최대 노드 {max_nodes:,} 가 int16 한계 32,767 을 넘는다. "
            f"edge_index 를 int32 로 바꿀 것 (§2.8)")
        assert edge_index.size == 0 or edge_index.min() >= 0, \
            "edge_index 에 음수가 있다 — int16 오버플로가 이미 발생했다 (§2.8)"

    arrays = dict(
        node_feat=np.concatenate(feats, axis=0),
        node_ptr=node_ptr,
        edge_index=edge_index,
        edge_ptr=edge_ptr,
        scalars=np.stack(scalars),
        labels=np.array(labels, dtype=np.int64),
        lot_ids=np.array(lots, dtype=np.int64),
        dup_ids=dup_ids,
        drop_log=np.array([drop_log[c] for c in DROP_CODES], dtype=np.int64),
        theta_bins=theta_bins,
        r_bins=r_bins,
    )
    if include_polar:
        arrays["polar"] = np.stack(polars)
    np.savez_compressed(output_npz, **arrays)

    n = len(labels)
    n_groups = len(np.unique(dup_ids))
    print(f"\n완료: {n}개 저장, {sum(drop_log.values())}개 제외, {time.time() - t0:.0f}s")
    print("배제 사유별 (§1.3):", ", ".join(f"{c} {drop_log[c]}" for c in DROP_CODES))
    print(f"중복 그룹 (§1.4): {n_groups:,}개 / 여분 웨이퍼 {n - n_groups}장")
    print(f"평균 노드 수 {node_ptr[-1] / n:.1f}, 평균 방향성 엣지 수 {edge_ptr[-1] / n:.1f}")
    print(f"극좌표 이미지: {'포함' if include_polar else '미포함(graph 전용)'}")
    print(f"산출물 해시 (§2.8): {file_hash(output_npz)}")
    print(f"-> {output_npz}")
    return output_npz


def _drop_reason(wafer_map, min_valid_die, min_defect_die):
    """§1.3 배제 사유를 판정한다. 해당 없으면 None."""
    m = np.asarray(wafer_map)
    n_valid = int((m > 0).sum())
    if n_valid == 0:
        return "no_valid_die"
    n_defect = int((m == 2).sum())
    if n_defect == 0:
        return "no_defect"
    if n_valid < min_valid_die or n_defect < min_defect_die:
        return "too_small"
    vy, vx = np.nonzero(m > 0)
    if float(np.hypot(vx - vx.mean(), vy - vy.mean()).max()) <= 0:
        return "degenerate"
    return None


# 4. 증강 · Dataset · 로더


In [ ]:
"""전처리 결과를 PyG 데이터로 공급하는 Dataset + 회전/반전 증강.

[개선 4] 데이터 증강
웨이퍼 결함 라벨은 웨이퍼를 돌리거나 뒤집어도 바뀌지 않는다
(Center/Donut/Edge-Ring/Random/Near-full 은 회전 불변, Loc/Edge-Loc/Scratch 도
반경 위치와 형태가 유지되므로 라벨이 보존된다).
극좌표 표현에서 회전은 theta 축 순환 시프트 한 줄이라 비용이 사실상 0이다.

중요: 그래프 트랙과 이미지 트랙에 **동일한** 변환을 적용해야 한다.
서로 다른 각도로 회전하면 두 트랙이 모순된 정보를 주게 되어 융합이 망가진다.
"""


import numpy as np
import torch
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected



class PackedWaferData:
    """`build_dataset.py` 가 만든 .npz 를 메모리에 올린다.

    노드/엣지는 가변 길이라 하나의 큰 배열 + 오프셋(ptr) 형태로 저장한다.
    float16 / int16 로 보관하고 꺼낼 때만 캐스팅해 RAM 사용량을 4분의 1로 줄인다.
    """

    def __init__(self, npz_path: str):
        z = np.load(npz_path, allow_pickle=False)
        self.node_feat = z["node_feat"]      # [총 노드 수, F] float16
        self.node_ptr = z["node_ptr"]        # [N+1]
        self.edge_index = z["edge_index"]    # [2, 총 엣지 수] int16 (그래프 로컬 인덱스)
        self.edge_ptr = z["edge_ptr"]        # [N+1]
        # graph 전용 npz(build_dataset(..., include_polar=False))에는 polar 가 없다.
        self.polar = z["polar"] if "polar" in z.files else None
        self.scalars = z["scalars"]          # [N, S] float32
        self.labels = z["labels"]            # [N] int64
        self.lot_ids = z["lot_ids"]          # [N] int64 (lot 단위 분할용)
        # §1.4 중복 웨이퍼 그룹. 없는 예전 파일은 웨이퍼마다 고유 그룹으로 둔다
        # (그룹 제약이 아무것도 묶지 않는 상태 = 기존 동작과 동일).
        self.dup_ids = (z["dup_ids"] if "dup_ids" in z.files
                        else np.arange(len(self.labels), dtype=np.int64))
        self.drop_log = z["drop_log"] if "drop_log" in z.files else None
        self.theta_bins = int(z["theta_bins"])

    def __len__(self):
        return len(self.labels)

    def get(self, i: int):
        ns, ne = self.node_ptr[i], self.node_ptr[i + 1]
        es, ee = self.edge_ptr[i], self.edge_ptr[i + 1]
        polar = (torch.from_numpy(self.polar[i].astype(np.float32))
                 if self.polar is not None else None)
        return (
            torch.from_numpy(self.node_feat[ns:ne].astype(np.float32)),
            torch.from_numpy(self.edge_index[:, es:ee].astype(np.int64)),
            polar,
            torch.from_numpy(self.scalars[i]),
            int(self.labels[i]),
        )


class WaferDualTrackDataset(torch.utils.data.Dataset):
    """PyG `Data` 를 돌려주는 Dataset. `augment=True` 일 때만 증강한다.

    Parameters
    ----------
    augment : 학습 세트에만 True. 검증/테스트에는 반드시 False.
    rotate : theta 빈 단위 무작위 회전.
    flip : 무작위 좌우 반전(거울상).
    """

    def __init__(self, packed: PackedWaferData, indices, augment=False,
                 rotate=True, flip=True, seed=0):
        self.packed = packed
        self.indices = np.asarray(indices)
        self.augment = augment
        self.rotate = rotate
        self.flip = flip
        self.theta_bins = packed.theta_bins
        self._seed = seed
        self._rng = np.random.default_rng(seed)

    def reset_rng(self, seed=None):
        """증강 난수를 초기 상태로 되돌린다.

        로더 하나를 여러 모델이 공유하면 난수 스트림이 이어진다. 첫 모델이 5000개를
        쓰면 두 번째 모델은 5001번부터 시작해, 서로 다른 증강 데이터를 보게 된다.
        학습을 시작하기 전에 이 함수를 불러야 모든 모델이 같은 증강을 본다.
        """
        self._seed = self._seed if seed is None else seed
        self._rng = np.random.default_rng(self._seed)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        x, edge_index, polar, scalars, y = self.packed.get(self.indices[i])

        if self.augment:
            if self.flip and self._rng.random() < 0.5:
                x, polar = _apply_flip(x, polar)
            if self.rotate:
                shift = int(self._rng.integers(0, self.theta_bins))
                if shift:
                    x, polar = _apply_rotation(x, polar, shift, self.theta_bins)

        # 저장은 방향성 k-NN 엣지로 하고, 여기서 대칭화한다(중복 제거 포함).
        if edge_index.numel():
            edge_index = to_undirected(edge_index, num_nodes=x.size(0))

        data = Data(
            x=x,
            edge_index=edge_index,
            scalars=scalars.unsqueeze(0),      # [1, S] -> [B, S]
            y=torch.tensor([y], dtype=torch.long),
        )
        if polar is not None:
            data.polar = polar.unsqueeze(0)    # [1, C, R, T] -> 배치 시 [B, C, R, T]
        return data


# ----------------------------------------------------------------------
# 증강 연산: 두 트랙이 정확히 같은 변환을 받도록 한 곳에서 정의한다.
# ----------------------------------------------------------------------
def _apply_rotation(x: torch.Tensor, polar: torch.Tensor | None, shift: int, theta_bins: int):
    """theta 축으로 `shift` 빈 회전.

    극좌표 인덱스가 ti = ((theta + pi) / 2pi * T) % T 이므로 점을
    +phi 만큼 돌리면 ti -> ti + shift 가 된다. 이는 torch.roll 과 정확히 일치한다.

    polar 가 None(graph 전용 데이터)이면 그래프 쪽만 회전한다.
    """
    phi = 2.0 * np.pi * shift / theta_bins
    c, s = np.cos(phi), np.sin(phi)
    x = x.clone()
    px, py = x[:, FEAT_X].clone(), x[:, FEAT_Y].clone()
    x[:, FEAT_X] = c * px - s * py
    x[:, FEAT_Y] = s * px + c * py
    cx, sy = x[:, FEAT_COS].clone(), x[:, FEAT_SIN].clone()
    x[:, FEAT_COS] = c * cx - s * sy
    x[:, FEAT_SIN] = s * cx + c * sy
    if polar is None:
        return x, None
    return x, torch.roll(polar, shifts=shift, dims=-1)


def _apply_flip(x: torch.Tensor, polar: torch.Tensor | None):
    """y -> -y 거울상. 극좌표에서는 theta -> -theta.

    u = (theta + pi) / 2pi * T 이고 비닝이 round(u) 이므로 반전 후에는
    round(T - u) = T - round(u), 즉 ti -> (-ti) % T 가 된다.
    torch.flip 은 i -> T-1-i 이라 한 칸 모자라므로 roll(+1) 로 보정한다.
    """
    x = x.clone()
    x[:, FEAT_Y] = -x[:, FEAT_Y]
    x[:, FEAT_SIN] = -x[:, FEAT_SIN]
    if polar is None:
        return x, None
    return x, torch.roll(torch.flip(polar, dims=[-1]), shifts=1, dims=-1)


# ----------------------------------------------------------------------
# 학습 코드에 바로 물릴 수 있는 로더 구성 헬퍼.
# 모델은 포함하지 않는다 -- 어떤 graph 모델이든 이 로더를 그대로 받아 쓰면 된다.
# ----------------------------------------------------------------------
def make_splits(labels, lot_ids, mode="stratified", holdout_test=False, seed=42,
                fold=None, n_folds=5):
    """(train, val, test) 인덱스를 만든다.

    mode="stratified" : v1 과 동일한 무작위 층화 분할. 같은 lot 이 양쪽에 섞인다.
    mode="lot"        : 같은 lot 은 train/val/test 중 한 곳에만. 누수가 사라진다.
    holdout_test=False: test = val (v1 호환). True 면 70/15/15 3분할.

    fold 를 주면 **k-fold 교차검증** 모드가 되고 holdout_test 는 무시된다.
    전체를 n_folds 조각으로 나눠 fold 번째 조각을 test 로 쓴다. 나머지에서 다시
    떼어 val 을 만든다(early stopping 용). n_folds=5 면 train 70 / val 10 / test 20.

    왜 CV 인가: 단일 분할은 test 가 전체의 15~20% 뿐이라 희소 클래스 표본이 너무
    적다. Near-full 은 전체 149장 중 test 에 22장만 들어가 한 장만 틀려도 recall 이
    0.045 움직인다. 이 표본 잡음만으로 macro F1 이 +-0.010 흔들린다. CV 는
    fold 를 이어붙이면 모든 웨이퍼가 정확히 한 번씩 채점되므로 이 잡음이 사라진다.
    """
    from sklearn.model_selection import (
        GroupShuffleSplit,
        StratifiedKFold,
        train_test_split,
    )

    idx = np.arange(len(labels))

    def split(indices, frac):
        if mode == "lot":
            gss = GroupShuffleSplit(n_splits=1, test_size=frac, random_state=seed)
            a, b = next(gss.split(indices, labels[indices], lot_ids[indices]))
        else:
            a, b = train_test_split(np.arange(len(indices)), test_size=frac,
                                    stratify=labels[indices], random_state=seed)
        return indices[a], indices[b]

    if fold is not None:
        if not 0 <= fold < n_folds:
            raise ValueError(f"fold 는 0 이상 {n_folds} 미만이어야 한다: {fold}")
        if mode == "lot":
            from sklearn.model_selection import StratifiedGroupKFold
            kf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
            parts = list(kf.split(idx, labels, lot_ids))
        else:
            kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
            parts = list(kf.split(idx, labels))
        rest, test_idx = parts[fold]
        # 남은 (n_folds-1)/n_folds 에서 val 을 떼어낸다. n_folds=5 면 0.125 -> 전체의 10%.
        train_idx, val_idx = split(idx[rest], 1.0 / (2 * (n_folds - 1)))
        return train_idx, val_idx, idx[test_idx]

    if not holdout_test:
        train_idx, val_idx = split(idx, 0.2)
        return train_idx, val_idx, val_idx
    train_idx, rest = split(idx, 0.3)
    val_idx, test_idx = split(rest, 0.5)
    return train_idx, val_idx, test_idx


def _worker_init(worker_id):
    """DataLoader 워커마다 증강 RNG 를 다시 뿌린다(안 하면 모든 워커가 같은 난수를 쓴다)."""
    info = torch.utils.data.get_worker_info()
    info.dataset._rng = np.random.default_rng(torch.initial_seed() % (2**32))


def make_loaders(npz_path, batch_size=64, augment=True, split="stratified",
                 holdout_test=False, seed=42, num_workers=2, balanced_sampler=False,
                 split_seed=None, fold=None, n_folds=5, packed=None):
    """npz 하나로 (train_loader, val_loader, test_loader, packed) 를 만든다.

    증강은 train 에만 걸린다. val/test 에 증강이 들어가면 채점 문제 자체가 바뀐다.

    seed 와 split_seed 를 분리한다.
      split_seed : train/val/test 를 어떻게 자를지 결정. **모든 실험에서 고정**해야
                   모델들이 같은 시험지로 채점돼 비교가 짝지어진다.
      seed       : 모델 초기화·셔플·증강 등 학습 무작위성만 결정.
    split_seed 를 주지 않으면 seed 와 같아진다(기존 동작 유지).

    fold 를 주면 k-fold 교차검증의 fold 번째 분할을 쓴다.
    packed 를 주면 .npz 를 다시 읽지 않고 재사용한다(로더를 여러 벌 만들 때).

    balanced_sampler=True 면 학습 배치를 클래스 균형에 맞춰 뽑는다.
    현재 Near-full 은 train 119장뿐이라 배치(64)의 69% 에 한 장도 안 들어간다.
    복원 추출로 소수 클래스를 자주 뽑으면 배치마다 고르게 등장한다. 증강이
    on-the-fly 라 같은 웨이퍼가 여러 번 뽑혀도 매번 다른 각도로 나온다.
    epoch 당 표본 수는 그대로라 학습 시간은 변하지 않는다.

    주의: 이걸 켜면 손실 함수의 클래스 가중치는 끄거나 완화해야 한다.
    둘 다 걸면 소수 클래스가 이중으로 증폭돼 precision 이 무너진다.

    사용 예)
        tr, va, te, packed = make_loaders("wafer_graph.npz")
        model = MyGraphNet(in_dim=NUM_NODE_FEATURES, num_classes=8)
        for batch in tr:
            out = model(batch.x, batch.edge_index, batch.batch)
    """
    from torch.utils.data import WeightedRandomSampler
    from torch_geometric.loader import DataLoader

    packed = PackedWaferData(npz_path) if packed is None else packed
    train_idx, val_idx, test_idx = make_splits(
        packed.labels, packed.lot_ids, split, holdout_test,
        seed if split_seed is None else split_seed, fold=fold, n_folds=n_folds)

    train_set = WaferDualTrackDataset(packed, train_idx, augment=augment, seed=seed)
    val_set = WaferDualTrackDataset(packed, val_idx, augment=False)
    test_set = WaferDualTrackDataset(packed, test_idx, augment=False)

    # persistent_workers 를 켜면 워커가 한 번만 생성되어 worker_init_fn 도 한 번만 돈다.
    # 그러면 워커 안의 증강 난수를 학습마다 되돌릴 수 없어 재현이 깨진다.
    # 매 epoch 워커를 다시 만드는 비용(약 1초)을 내고 재현성을 얻는다.
    common = dict(num_workers=num_workers, persistent_workers=False)

    # 데이터 순서를 모델 초기화와 분리한다. 전역 난수를 쓰면 파라미터 수가 다른 모델
    # (GIN 177K vs GAT 128K)이 초기화에서 난수를 다르게 소비해 셔플 순서까지 달라진다.
    loader_gen = torch.Generator().manual_seed(seed)
    if balanced_sampler:
        # 표본별 가중치 = 그 표본이 속한 클래스의 가중치. 복원 추출로 소수 클래스를 자주 뽑는다.
        cw = class_weights(packed.labels[train_idx])
        sample_w = torch.as_tensor(cw[packed.labels[train_idx]], dtype=torch.double)
        sampler = WeightedRandomSampler(sample_w, num_samples=len(train_idx),
                                        replacement=True, generator=loader_gen)
        train_loader = DataLoader(train_set, batch_size=batch_size, sampler=sampler,
                                  generator=loader_gen,
                                  worker_init_fn=_worker_init if num_workers else None,
                                  **common)
    else:
        train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                                  generator=loader_gen,
                                  worker_init_fn=_worker_init if num_workers else None,
                                  **common)
    train_loader._seed = seed          # train_one 에서 되돌리기 위해 보관
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, **common)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, **common)
    where = f"fold {fold}/{n_folds}" if fold is not None else f"holdout={holdout_test}"
    print(f"train {len(train_idx)} / val {len(val_idx)} / test {len(test_idx)}"
          f"  (분할: {split}, {where}, 증강: {augment}, 균형샘플러: {balanced_sampler})")
    return train_loader, val_loader, test_loader, packed


def loaders_from_indices(packed, train_idx, val_idx, eval_idx=None, batch_size=64,
                         augment=True, seed=42, num_workers=2, balanced_sampler=False,
                         verbose=True):
    """§3.5 로 고정된 인덱스를 그대로 받아 로더를 만든다.

    `make_loaders` 와 달리 분할을 **계산하지 않는다.** 사양서는 분할을 파일로
    고정하고 모든 실험이 그것을 로드하도록 요구하므로, 학습 코드가 인덱스를
    다시 만들 여지를 없앤다.

    eval_idx 를 주지 않으면 검증 집합을 그대로 쓴다. 개발 단계에서는 그게 맞다 --
    §3.3 대로 검증 지표는 조기 종료 편향을 포함하며 **상대 비교 전용**이다.
    최종 평가는 별도 경로(`final_evaluation`)로만 수행한다.
    """
    from torch.utils.data import WeightedRandomSampler
    from torch_geometric.loader import DataLoader

    eval_idx = val_idx if eval_idx is None else eval_idx
    train_set = WaferDualTrackDataset(packed, train_idx, augment=augment, seed=seed)
    val_set = WaferDualTrackDataset(packed, val_idx, augment=False)
    eval_set = WaferDualTrackDataset(packed, eval_idx, augment=False)

    common = dict(num_workers=num_workers, persistent_workers=False)
    loader_gen = torch.Generator().manual_seed(seed)
    if balanced_sampler:
        # §4.2 클래스 가중치는 이때 꺼야 한다(이중 증폭 방지). 가중치는 §4.1 대로
        # **해당 fold 의 학습 분할에서만** 산출한다.
        cw = class_weights(packed.labels[train_idx])
        sample_w = torch.as_tensor(cw[packed.labels[train_idx]], dtype=torch.double)
        sampler = WeightedRandomSampler(sample_w, num_samples=len(train_idx),
                                        replacement=True, generator=loader_gen)
        train_loader = DataLoader(train_set, batch_size=batch_size, sampler=sampler,
                                  generator=loader_gen,
                                  worker_init_fn=_worker_init if num_workers else None,
                                  **common)
    else:
        train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                                  generator=loader_gen,
                                  worker_init_fn=_worker_init if num_workers else None,
                                  **common)
    train_loader._seed = seed
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, **common)
    eval_loader = DataLoader(eval_set, batch_size=batch_size, shuffle=False, **common)
    if verbose:
        print(f"train {len(train_idx)} / val {len(val_idx)} / eval {len(eval_idx)}"
              f"  (증강: {augment}, 균형샘플러: {balanced_sampler})")
    return train_loader, val_loader, eval_loader


def class_weights(labels, num_classes=8, alpha=1.0):
    """sklearn 의 'balanced' 와 같은 식. 클래스가 빠져도 길이를 항상 보장한다.

    alpha 는 가중치의 세기다. w = (N / (C * n_c)) ** alpha.

    alpha=1.0 이면 완전 보정(기본). WM-811K 는 불균형이 65:1 이라 Near-full 의
    벌점이 Edge-Ring 의 65배가 되는데, 실측 결과 이게 **과했다** --
    소수 클래스 재현율은 0.94~1.00 인데 정밀도가 0.71~0.79 로 무너지고,
    그 대가로 Loc 이 274장을 잃었다(macro 재현율 0.914 vs 정밀도 0.833).

    alpha=0.5 면 벌점비가 65:1 -> 8:1 로 완화된다. alpha=0 이면 가중치 없음.
    F1 은 정밀도와 재현율이 비슷할 때 최대이므로, 한쪽으로 치우친 현재 지점은
    양쪽 모두 손해다.
    """
    counts = np.bincount(labels, minlength=num_classes).astype(np.float64)
    w = np.where(counts > 0, len(labels) / (num_classes * np.maximum(counts, 1)), 0.0)
    return w ** alpha if alpha != 1.0 else w


# 5. 분할 모듈


In [ ]:
"""사양서 §3 — 봉인 2단계 분할.

    결함 데이터
    ├─ 개발 구간 70%  ─── 5-fold (각 fold: 학습 4/5, 검증 1/5)
    └─ 최종 평가 30%  ─── 봉인

세 가지를 **동시에** 만족해야 한다.

1. 층화   — failureType 기준. 빼면 Near-full(149장)이 한쪽으로 쏠린다.
2. 그룹   — 같은 dup_ids 는 어떤 경계도 넘지 않는다. 중복 웨이퍼 93그룹은
            동일 로트 내 중복이 0% 라 로트 분할로 차단되지 않는 독립 누수 채널이다.
3. 봉인   — 최종 평가 집합은 어떤 선택에도 관여하지 않는다.

층화와 그룹을 동시에 지키는 것이 핵심이라 `StratifiedGroupKFold` 를 쓴다.
`GroupShuffleSplit` 은 층화가 없고 `train_test_split(stratify=)` 는 그룹이 없어
둘 다 단독으로는 요건을 못 채운다.

분할은 한 번 만들어 파일로 고정하고(§3.5) 이후 모든 실험이 이를 로드한다.
재계산하면 라이브러리 버전 차이만으로도 경계가 달라질 수 있다.
"""


import json
import os

import numpy as np

NUM_CLASSES = 8

# §3.4 경고 조건 임계값
MIN_SUPPORT_WARN = 10


def make_sealed_splits(labels, groups, dev_frac=0.7, n_folds=5, seed=42,
                       outer_splits=20):
    """§3.1 개발/최종 2단계 분할을 만든다.

    Parameters
    ----------
    labels : [W] 클래스 인덱스. 층화 기준.
    groups : [W] dup_ids. 같은 값끼리는 절대 갈라지지 않는다.
    dev_frac : 개발 구간 비율. 0.7 이면 개발 70 / 최종 30.
    outer_splits : 1단계를 몇 조각으로 나눠 그중 몇 개를 최종 평가로 뗄지 결정한다.
        20 이면 0.7 -> 6/20 = 정확히 30%. 10 으로 하면 3/10 = 30% 지만
        0.75 같은 값에서 정확도가 떨어진다.

    Returns
    -------
    dict : {"dev": [...], "final": [...], "folds": [(train, val), ...]}
    """
    from sklearn.model_selection import StratifiedGroupKFold

    labels = np.asarray(labels)
    groups = np.asarray(groups)
    idx = np.arange(len(labels))

    n_final = int(round(outer_splits * (1.0 - dev_frac)))
    if not 1 <= n_final < outer_splits:
        raise ValueError(f"dev_frac={dev_frac} 이 outer_splits={outer_splits} 와 맞지 않는다")

    # --- 1단계: 개발 / 최종 평가 -------------------------------------------
    outer = StratifiedGroupKFold(n_splits=outer_splits, shuffle=True, random_state=seed)
    parts = [te for _, te in outer.split(idx, labels, groups)]
    final_idx = np.sort(np.concatenate(parts[:n_final]))
    dev_idx = np.sort(np.concatenate(parts[n_final:]))

    # --- 2단계: 개발 구간 안에서 5-fold ------------------------------------
    inner = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    folds = []
    for tr, va in inner.split(dev_idx, labels[dev_idx], groups[dev_idx]):
        folds.append((np.sort(dev_idx[tr]), np.sort(dev_idx[va])))

    return {"dev": dev_idx, "final": final_idx, "folds": folds}


def validate_splits(split, labels, groups, num_classes=NUM_CLASSES, verbose=True):
    """§3.4 분할 직후 필수 검증.

    중단 조건을 하나라도 위반하면 AssertionError 를 낸다. 경고 조건은 목록으로
    돌려주고 진행은 막지 않는다.

    Returns
    -------
    (warnings, report) : 경고 문자열 목록과 클래스별 지원 수 표(dict).
    """
    labels = np.asarray(labels)
    groups = np.asarray(groups)
    dev, final, folds = split["dev"], split["final"], split["folds"]

    def _(a):
        return set(np.asarray(a).tolist())

    # --- 중단 조건 ---------------------------------------------------------
    assert not (_(dev) & _(final)), "개발 구간과 최종 평가가 겹친다"

    for i, (tr, va) in enumerate(folds):
        assert not (_(tr) & _(va)), f"fold {i}: 학습과 검증이 겹친다"
        assert not (_(tr) & _(final)), f"fold {i}: 학습이 최종 평가와 겹친다"
        assert not (_(va) & _(final)), f"fold {i}: 검증이 최종 평가와 겹친다"
        assert _(tr) | _(va) <= _(dev), f"fold {i}: 개발 구간 밖의 인덱스가 있다"

    # 그룹 무결성 — 어떤 dup_ids 그룹도 경계를 넘지 않는다
    _assert_group_intact(groups, [("개발", dev), ("최종평가", final)])
    for i, (tr, va) in enumerate(folds):
        _assert_group_intact(groups, [(f"fold{i}학습", tr), (f"fold{i}검증", va)])

    # 8개 클래스가 각 fold 검증과 최종 평가에 1장 이상
    for name, part in [("최종 평가", final)] + [(f"fold {i} 검증", va)
                                                for i, (_t, va) in enumerate(folds)]:
        cnt = np.bincount(labels[part], minlength=num_classes)
        missing = np.where(cnt == 0)[0].tolist()
        assert not missing, f"{name} 에 없는 클래스가 있다: {missing}"

    # --- 경고 조건 ---------------------------------------------------------
    warns, report = [], {}
    for i, (_t, va) in enumerate(folds):
        cnt = np.bincount(labels[va], minlength=num_classes)
        report[f"fold{i}_val"] = cnt
        for c in np.where(cnt < MIN_SUPPORT_WARN)[0]:
            warns.append(f"fold {i} 검증에서 클래스 {c} 지원 수 {cnt[c]} (<{MIN_SUPPORT_WARN}) "
                         f"— 웨이퍼 1장당 재현율 {100 / cnt[c]:.1f}%p 변동")
    report["final"] = np.bincount(labels[final], minlength=num_classes)
    report["dev"] = np.bincount(labels[dev], minlength=num_classes)

    if verbose:
        print(f"[§3.4] 중단 조건 전항 통과 "
              f"(개발 {len(dev):,} / 최종평가 {len(final):,} / fold {len(folds)}개)")
        for w in warns:
            print("  [경고]", w)
    return warns, report


def _assert_group_intact(groups, parts):
    """같은 그룹 id 가 두 파트에 걸쳐 있지 않은지 확인한다."""
    seen = {}
    for name, part in parts:
        for g in np.unique(groups[np.asarray(part)]):
            if g in seen and seen[g] != name:
                raise AssertionError(
                    f"dup_ids 그룹 {g} 가 '{seen[g]}' 와 '{name}' 로 갈라졌다")
            seen[g] = name


def save_splits(split, path, meta):
    """§3.5 분할 고정. 인덱스와 메타데이터를 함께 저장한다.

    meta 에는 분할 모드·시드·전처리 산출물 해시를 반드시 넣는다. 해시가 없으면
    전처리가 바뀐 뒤에도 옛 분할을 그대로 쓰는 사고를 막을 수 없다.
    """
    for k in ("mode", "seed", "preproc_hash"):
        if k not in meta:
            raise ValueError(f"meta 에 '{k}' 가 없다 (§3.5)")
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    arrays = {"dev": split["dev"], "final": split["final"],
              "n_folds": np.array(len(split["folds"]))}
    for i, (tr, va) in enumerate(split["folds"]):
        arrays[f"fold{i}_train"] = tr
        arrays[f"fold{i}_val"] = va
    arrays["meta"] = np.frombuffer(json.dumps(meta).encode(), dtype=np.uint8)
    np.savez_compressed(path, **arrays)
    print(f"[§3.5] 분할 저장: {path}")
    return path


def load_splits(path, preproc_hash=None):
    """저장된 분할을 읽는다. preproc_hash 를 주면 일치 여부를 강제한다."""
    z = np.load(path, allow_pickle=False)
    meta = json.loads(bytes(z["meta"]).decode())
    if preproc_hash is not None and meta.get("preproc_hash") != preproc_hash:
        raise ValueError(
            f"전처리 해시 불일치 — 분할 파일은 {meta.get('preproc_hash')}, "
            f"현재 데이터는 {preproc_hash}. 전처리가 바뀌었으므로 분할을 다시 만들어야 한다.")
    n = int(z["n_folds"])
    split = {"dev": z["dev"], "final": z["final"],
             "folds": [(z[f"fold{i}_train"], z[f"fold{i}_val"]) for i in range(n)]}
    return split, meta


def split_report(split, labels, class_names, num_classes=NUM_CLASSES):
    """분할 구성을 표로 만든다(pandas DataFrame)."""
    import pandas as pd

    labels = np.asarray(labels)
    rows = []
    for name, part in ([("개발", split["dev"]), ("최종평가", split["final"])]
                       + [(f"fold{i} 검증", va) for i, (_t, va) in enumerate(split["folds"])]):
        cnt = np.bincount(labels[part], minlength=num_classes)
        rows.append({"구간": name, "장수": len(part),
                     **{class_names[i]: int(cnt[i]) for i in range(num_classes)}})
    return pd.DataFrame(rows)


# 6. 그래프 모델


In [ ]:
"""GCN / GraphSAGE / GAT / GIN 비교용 통합 모델.

**공정한 비교를 위해 conv 계층 하나만 바뀌고 나머지는 전부 동일하다.**
백본, 층 수, hidden 차원, BatchNorm, residual, dropout, readout, 분류기 MLP,
스칼라 주입까지 모두 공유한다. 그래야 성능 차이를 conv 종류의 차이로 해석할 수 있다.

conv 별 차이

- GCN       이웃 정보를 차수로 정규화해 평균낸다. 가장 단순하고 파라미터가 적다.
- GraphSAGE 자기 자신(root)과 이웃 집계를 **분리된 가중치**로 다룬다.
            이웃에 묻히지 않아 Scratch 처럼 이웃이 2개뿐인 구조에 유리할 수 있다.
- GAT       어텐션으로 이웃마다 가중치를 다르게 준다. 불규칙한 형태에 강하다고 알려져 있다.
            heads 개로 나눠 계산하되 출력 차원은 hidden 으로 맞춘다.
- GIN       합(sum) 집계 + MLP. 이론적 표현력이 가장 높고 그래프 구조 구분에 강하다.
            합 집계라 노드 수(=결함 개수)에 민감하다 -- Near-full 에 유리할 수 있다.

네 가지 모두 PyG 에서 `conv(x, edge_index)` 시그니처가 같아 그대로 교체된다.
"""


import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import (
    GATConv,
    GCNConv,
    GINConv,
    SAGEConv,
    global_add_pool,
    global_max_pool,
    global_mean_pool,
)


CONV_NAMES = ("gcn", "sage", "gat", "gin")

# 메시지 전달을 좌표 그래프가 아니라 임베딩 공간에서 매번 다시 만드는 방식(논문 방식).
# 그리고 메시지 전달을 아예 하지 않는 베이스라인.
EXTRA_CONV_NAMES = ("dyn", "none")
ALL_CONV_NAMES = CONV_NAMES + EXTRA_CONV_NAMES

# forward 시그니처가 conv(x, edge_index) 가 아닌 것들
DYNAMIC_CONVS = {"dyn"}     # conv(x, batch) — 이웃을 스스로 찾는다
NO_MESSAGE_CONVS = {"none"} # 이웃을 아예 보지 않는다

DISPLAY_NAME = {
    "gcn": "GCN",
    "sage": "GraphSAGE",
    "gat": "GAT",
    "gin": "GIN",
    "dyn": "DynEdge",
    "none": "NoMsg(기준선)",
}


def make_conv(name: str, in_dim: int, out_dim: int, heads: int = 4, knn_k: int = 8):
    """이름으로 conv 계층 하나를 만든다. 출력 차원은 항상 out_dim 으로 맞춘다."""
    if name == "gcn":
        return GCNConv(in_dim, out_dim)
    if name == "sage":
        return SAGEConv(in_dim, out_dim)
    if name == "gat":
        # heads 개로 쪼개 concat 하므로 head 당 out_dim // heads 차원.
        # 이렇게 해야 다른 conv 와 출력 차원이 같아져 비교가 성립한다.
        if out_dim % heads != 0:
            raise ValueError(f"hidden({out_dim})이 heads({heads})로 나누어떨어져야 한다")
        return GATConv(in_dim, out_dim // heads, heads=heads, concat=True)
    if name == "gin":
        # GIN 은 집계 후 MLP 를 태우는 것이 정의의 일부다.
        mlp = nn.Sequential(
            nn.Linear(in_dim, out_dim), nn.ReLU(inplace=True), nn.Linear(out_dim, out_dim)
        )
        return GINConv(mlp, train_eps=True)
    if name == "dyn":
        return DynamicKNNConv(in_dim, out_dim, k=knn_k)
    if name == "none":
        # 메시지 전달 없음. 각 노드를 이웃과 무관하게 독립적으로 변환한다.
        # 그래프 구조가 실제로 기여하는지 재는 기준선이다(DeepSets 형태).
        return nn.Sequential(
            nn.Linear(in_dim, out_dim), nn.ReLU(inplace=True), nn.Linear(out_dim, out_dim)
        )
    raise ValueError(f"알 수 없는 conv: {name} (가능: {ALL_CONV_NAMES})")


class DynamicKNNConv(nn.Module):
    """[논문 방식] 임베딩 공간에서 매 층 k-NN 을 다시 만드는 EdgeConv.

    좌표 k-NN 과 달리 웨이퍼 반대편에 떨어진 두 결함 덩어리도 '비슷하게 생겼으면'
    연결된다. Scratch(끊긴 선)나 Edge-Loc(가장자리 여러 군데)처럼 떨어진 조각들이
    하나의 패턴을 이루는 경우에 유리할 수 있다.

    PyG 의 DynamicEdgeConv 는 pyg-lib/torch-cluster 를 요구하는데 Colab 의
    `pip install torch_geometric` 만으로는 설치되지 않는다. 그래서 순수 PyTorch 로 짰다.
    배치 안의 그래프들은 메모리상 연속이므로 split 으로 잘라 그래프별로 처리한다
    (그래프 경계를 넘어 이웃을 찾으면 안 되기 때문).
    """

    def __init__(self, in_dim: int, out_dim: int, k: int = 8):
        super().__init__()
        self.k = k
        self.out_dim = out_dim
        # EdgeConv 는 (x_i, x_j - x_i) 를 이어붙여 넣으므로 입력이 2배다.
        self.mlp = nn.Sequential(
            nn.Linear(2 * in_dim, out_dim), nn.ReLU(inplace=True),
            nn.Linear(out_dim, out_dim),
        )

    def forward(self, x, batch):
        counts = torch.bincount(batch).tolist()
        outs = []
        for xi in torch.split(x, counts):
            n = xi.size(0)
            if n == 1:
                # 노드가 하나뿐이면 이웃이 없다. 자기 자신과의 차(=0)로 처리한다.
                pair = torch.cat([xi, torch.zeros_like(xi)], dim=-1)
                outs.append(self.mlp(pair))
                continue
            k = min(self.k, n - 1)
            d = torch.cdist(xi, xi)
            d.fill_diagonal_(float("inf"))        # 자기 자신은 이웃에서 제외
            idx = d.topk(k, dim=1, largest=False).indices      # [n, k]
            center = xi.unsqueeze(1).expand(-1, k, -1)         # [n, k, d]
            pair = torch.cat([center, xi[idx] - center], dim=-1)
            outs.append(self.mlp(pair).max(dim=1).values)      # max 집계
        return torch.cat(outs, dim=0)


class GraphClassifier(nn.Module):
    """conv 종류만 바꿔 끼우는 그래프 분류기.

    Parameters
    ----------
    conv : "gcn" | "sage" | "gat" | "gin" | "dyn" | "none"
    hidden : 은닉 차원. GAT 를 쓸 경우 heads 로 나누어떨어져야 한다.
    knn_k : "dyn" 전용. 임베딩 공간에서 몇 개를 이웃으로 볼지.
    num_layers : 메시지 전달 층 수.
    jk : Jumping Knowledge. None / "max" / "cat".
        None 이면 마지막 층 출력만 풀링에 쓴다(기본).
        측정 결과 클래스마다 필요한 시야가 정반대다 -- 3 hop 커버율과 메시지 전달
        효과의 상관이 r = -0.759 다. Random(커버 8%)은 더 깊어야 하고
        Scratch(커버 42%)는 오히려 얕아야 한다. JK 는 층별 출력을 모두 넘겨
        분류기가 클래스마다 어느 깊이를 쓸지 학습하게 한다.
        "max" 는 파라미터가 늘지 않고, "cat" 은 표현력이 크지만 약 +98K 늘어난다.
    use_scalars : 그래프 수준 스칼라 4개(결함 수, 결함 비율, 다이 수, 반경)를
        분류기에 직접 주입할지. 끄면 conv 자체의 성능만 본다.
    """

    def __init__(self, conv: str = "gcn", in_dim: int = NUM_NODE_FEATURES,
                 hidden: int = 128, num_layers: int = 3, dropout: float = 0.2,
                 head_dropout: float = 0.4, num_classes: int = 8, heads: int = 4,
                 use_scalars: bool = True, knn_k: int = 8, jk: str | None = None):
        super().__init__()
        if jk not in (None, "max", "cat"):
            raise ValueError(f"jk 는 None/'max'/'cat' 중 하나여야 한다: {jk}")
        self.conv_name = conv
        self.jk = jk
        self.use_scalars = use_scalars
        self.is_dynamic = conv in DYNAMIC_CONVS
        self.no_message = conv in NO_MESSAGE_CONVS

        self.input_proj = nn.Linear(in_dim, hidden)
        self.convs = nn.ModuleList(
            make_conv(conv, hidden, hidden, heads, knn_k) for _ in range(num_layers)
        )
        self.norms = nn.ModuleList(nn.BatchNorm1d(hidden) for _ in range(num_layers))
        self.dropout = dropout

        # readout: mean + max + log1p(sum) 3종 결합.
        # 평균만 쓰면 그래프 크기(=결함 개수)가 사라지는데, 이는 Near-full(평균 720개)과
        # Loc(평균 143개)을 가르는 핵심 정보다.
        # jk="cat" 이면 노드 특징이 hidden*num_layers 로 커지므로 readout 입력도 그만큼 커진다.
        node_dim = hidden * num_layers if jk == "cat" else hidden
        self.readout = nn.Sequential(
            nn.Linear(node_dim * 3, hidden), nn.BatchNorm1d(hidden), nn.ReLU(inplace=True)
        )

        clf_in = hidden + (NUM_GRAPH_SCALARS if use_scalars else 0)
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(clf_in),
            nn.Linear(clf_in, hidden), nn.ReLU(inplace=True), nn.Dropout(head_dropout),
            nn.Linear(hidden, hidden // 2), nn.ReLU(inplace=True), nn.Dropout(head_dropout),
            nn.Linear(hidden // 2, num_classes),
        )

    def forward(self, x, edge_index, batch, scalars=None):
        h = self.input_proj(x)
        layer_outs = []
        for conv, norm in zip(self.convs, self.norms):
            # conv 종류에 따라 호출 방식이 다르다.
            #   일반 GNN : conv(h, edge_index) — 전처리에서 만든 좌표 k-NN 그래프를 쓴다
            #   dyn      : conv(h, batch)      — 임베딩 공간에서 이웃을 직접 찾는다
            #   none     : conv(h)             — 이웃을 보지 않는다(기준선)
            if self.no_message:
                m = conv(h)
            elif self.is_dynamic:
                m = conv(h, batch)
            else:
                m = conv(h, edge_index)
            # residual: 층이 깊어질수록 노드 특징이 평준화되는 현상(over-smoothing)을 억제
            h = h + F.dropout(F.relu(norm(m)), self.dropout, self.training)
            layer_outs.append(h)

        # [JK] 마지막 층만 쓰지 않고 층별 출력을 결합한다.
        # 얕은 층 = 국소 정보(Scratch 용), 깊은 층 = 전역 정보(Random 용).
        if self.jk == "max":
            h = torch.stack(layer_outs, dim=0).max(dim=0).values
        elif self.jk == "cat":
            h = torch.cat(layer_outs, dim=1)

        h = F.relu(h)  # log1p 에 음수가 들어가지 않도록
        pooled = torch.cat(
            [
                global_mean_pool(h, batch),
                global_max_pool(h, batch),
                torch.log1p(global_add_pool(h, batch)),
            ],
            dim=1,
        )
        z = self.readout(pooled)
        if self.use_scalars:
            if scalars is None:
                raise ValueError("use_scalars=True 인데 scalars 가 없다")
            z = torch.cat([z, scalars], dim=1)
        return self.classifier(z)


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# 7. 이미지 트랙 · 게이트 융합 모델

**ResNet18 을 쓰지 않는다.** 파라미터가 11.2M 인데 fold 당 학습 데이터가
14,290장뿐이라 웨이퍼 1장당 780개다. v1 에서 val loss 가 0.483 -> 2.662 로
5.5배 폭증한 것이 정확히 이 문제였다.

| 모델 | 파라미터 |
|---|---:|
| GIN[base] (기준선) | 176,979 |
| PolarCNN 단독 | 191,216 |
| 게이트 융합 | 382,771 |
| ~~ResNet18~~ | ~~11,689,512~~ |

비교 대상과 용량이 비슷해야 **표현 방식의 차이**를 본 것이 되지 용량의 차이를
본 것이 되지 않는다.

**theta 축 순환 패딩**을 넣었다. 극좌표에서 theta = -pi 와 +pi 는 같은 위치인데
일반 conv 는 양 끝으로 찢어 버려, 경계에 걸친 Scratch/Edge-Loc 이 두 조각이 된다.
검증: `conv(roll(x)) == roll(conv(x))` 가 theta 축에서만 성립함을 확인했다.

> **미리 알아둘 한계**: 웨이퍼를 가로지르는 **직선 Scratch 는 극좌표에서 곡선**이 된다.
> 중심을 지나는 방사형만 직선으로 남는다. Scratch 가 현재 최하위(F1 0.782)인데
> 하필 그 클래스가 이 표현에서 불리하다. ①단계에서 드러날 것이다.


In [ ]:
"""극좌표 이미지 트랙과 그래프-이미지 게이트 융합.

왜 ResNet18 이 아닌가
---------------------
기존 `wafer_models.PolarResNet` 은 ResNet18(약 11.2M 파라미터)을 쓴다. 그런데
개발 구간 학습 데이터는 fold 당 14,290장뿐이다. **웨이퍼 1장당 파라미터 780개**로
과적합이 거의 확정적이며, v1 에서 val loss 가 0.483 -> 2.662 로 5.5배 폭증한 것이
정확히 이 문제였다.

그래서 여기서는 GIN(177K)과 규모를 맞춘 약 300K 짜리 CNN 을 쓴다. 비교 대상과
용량이 비슷해야 "표현 방식의 차이" 를 본 것이 되지 "용량의 차이" 를 본 것이 되지 않는다.

theta 축 순환 패딩
------------------
극좌표에서 theta = -pi 와 +pi 는 같은 위치인데 일반 conv 는 이미지 양 끝으로 찢어
버린다. 경계에 걸친 Scratch/Edge-Loc 이 두 조각이 된다. theta 축(마지막 축)에만
circular padding 을 준다.

알아둘 한계: 웨이퍼를 가로지르는 **직선 Scratch 는 극좌표에서 곡선이 된다.**
중심을 지나는 방사형만 직선으로 남는다. Scratch 가 현재 최하위(F1 0.782)인데
하필 그 클래스가 이 표현에서 불리하다. CNN 단독 성능을 먼저 재는 이유다.
"""


import torch
import torch.nn as nn
import torch.nn.functional as F



class CircularConvBlock(nn.Module):
    """conv -> BN -> ReLU. theta 축(마지막)만 순환 패딩, r 축은 zero 패딩."""

    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.conv = nn.Conv2d(cin, cout, kernel_size=3, stride=stride, padding=0, bias=False)
        self.bn = nn.BatchNorm2d(cout)

    def forward(self, x):
        x = F.pad(x, (1, 1, 0, 0), mode="circular")   # theta 축 순환
        x = F.pad(x, (0, 0, 1, 1), mode="constant")   # r 축은 경계가 실제 경계다
        return F.relu(self.bn(self.conv(x)), inplace=True)


class PolarCNN(nn.Module):
    """극좌표 3채널 [C, 64, 64] -> out_dim 임베딩. 약 300K 파라미터.

    채널: 0 결함 밀도 / 1 유효 다이 마스크 / 2 노이즈 판정 결함 밀도.
    """

    def __init__(self, in_channels=POLAR_CHANNELS, out_dim=128, width=32):
        super().__init__()
        w = width
        self.stem = CircularConvBlock(in_channels, w)
        self.blocks = nn.Sequential(
            CircularConvBlock(w, w), nn.MaxPool2d(2),            # 64 -> 32
            CircularConvBlock(w, w * 2), nn.MaxPool2d(2),        # 32 -> 16
            CircularConvBlock(w * 2, w * 2), nn.MaxPool2d(2),    # 16 -> 8
            CircularConvBlock(w * 2, w * 4), nn.MaxPool2d(2),    # 8  -> 4
        )
        # 평균만 쓰면 "어딘가에 강한 반응이 하나 있다"(Scratch)를 잃는다. max 를 함께 쓴다.
        self.head = nn.Sequential(
            nn.Linear(w * 4 * 2, out_dim), nn.BatchNorm1d(out_dim), nn.ReLU(inplace=True))

    def forward(self, polar):
        h = self.blocks(self.stem(polar))
        pooled = torch.cat([h.amax(dim=(2, 3)), h.mean(dim=(2, 3))], dim=1)
        return self.head(pooled)


class PolarCNNClassifier(nn.Module):
    """[1단계 진단] 이미지 트랙 **단독** 분류기.

    융합을 만들기 전에 이걸 먼저 돌린다. 융합이 이득을 보려면 두 트랙이 **서로 다른
    웨이퍼에서** 틀려야 하는데, 같은 곳에서 같이 틀리면 합쳐도 소용이 없다.
    단독 성능과 오라클 상한(`oracle_upper_bound`)을 보고 융합 진행 여부를 정한다.
    """

    needs_polar = True

    def __init__(self, num_classes=8, hidden=128, dropout=0.4, width=32,
                 use_scalars=True):
        super().__init__()
        self.use_scalars = use_scalars
        self.cnn = PolarCNN(out_dim=hidden, width=width)
        clf_in = hidden + (NUM_GRAPH_SCALARS if use_scalars else 0)
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(clf_in),
            nn.Linear(clf_in, hidden), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(hidden, num_classes),
        )

    def forward(self, x, edge_index, batch, scalars, polar):
        z = self.cnn(polar)
        if self.use_scalars:
            z = torch.cat([z, scalars], dim=1)
        return self.classifier(z)


class GatedFusion(nn.Module):
    """[3단계] 그래프 + 이미지 게이트 융합.

    웨이퍼마다 "그래프를 얼마나 볼지 / 이미지를 얼마나 볼지" 가중치를 학습한다.
    Cross-attention 보다 파라미터가 적어 14,290장 규모에서 안전하다.

        g = sigmoid(W [h_graph ; h_image])        # [B, hidden], 0~1
        z = g * h_graph + (1 - g) * h_image

    게이트를 스칼라 하나가 아니라 **차원별**로 두는 것이 중요하다. 어떤 특징은
    그래프가, 어떤 특징은 이미지가 나은데 스칼라 게이트는 그걸 표현하지 못한다.

    그래프 트랙은 확정된 `GIN[base]` 구성을 그대로 쓴다. 융합의 기여만 보려면
    그래프 쪽이 기준선과 동일해야 한다.
    """

    needs_polar = True

    def __init__(self, num_classes=8, hidden=128, conv="gin", num_layers=3,
                 heads=4, dropout=0.4, width=32, use_scalars=True):
        super().__init__()
        self.use_scalars = use_scalars
        # GraphClassifier 의 분류기 직전 표현을 쓰기 위해 마지막 분류층만 잘라낸다.
        self.graph = GraphClassifier(conv=conv, hidden=hidden, num_layers=num_layers,
                                     heads=heads, use_scalars=False)
        self.graph.classifier = nn.Identity()
        self.cnn = PolarCNN(out_dim=hidden, width=width)

        self.gate = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.Sigmoid())
        clf_in = hidden + (NUM_GRAPH_SCALARS if use_scalars else 0)
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(clf_in),
            nn.Linear(clf_in, hidden), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(hidden // 2, num_classes),
        )

    def forward(self, x, edge_index, batch, scalars, polar):
        hg = self.graph(x, edge_index, batch, None)   # use_scalars=False 라 None
        hi = self.cnn(polar)
        g = self.gate(torch.cat([hg, hi], dim=1))
        z = g * hg + (1 - g) * hi
        if self.use_scalars:
            z = torch.cat([z, scalars], dim=1)
        return self.classifier(z)

    @torch.no_grad()
    def gate_stats(self, x, edge_index, batch, scalars, polar):
        """게이트가 실제로 두 트랙을 섞는지 확인한다.

        평균이 0.5 근처면 두 트랙을 함께 쓰는 것이고, 0 이나 1 에 붙어 있으면
        한쪽을 사실상 버린 것이다 -- 그 경우 융합이 무의미하므로 반드시 확인한다.
        """
        hg = self.graph(x, edge_index, batch, None)
        hi = self.cnn(polar)
        g = self.gate(torch.cat([hg, hi], dim=1))
        return {"gate_mean": float(g.mean()), "gate_std": float(g.std()),
                "graph_dominant_dims": float((g > 0.5).float().mean())}


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# 8. 학습 · 평가 · 진단 함수


In [ ]:
"""GCN / GraphSAGE / GAT / GIN 을 같은 조건으로 학습해 비교한다.

공정성을 위해 지키는 것

1. **같은 분할** — seed 하나당 로더를 한 번만 만들어 모든 모델이 공유한다.
2. **같은 데이터** — train_one 이 시작할 때 전역 시드 · 증강 난수 · 로더 generator 를
   모두 되돌린다. 셋 다 되돌려야 모델 종류와 실행 순서에 무관하게 같은 데이터를 본다.
3. **같은 하이퍼파라미터** — hidden, 층 수, dropout, lr, epoch, 클래스 가중치 전부 공유.
4. **같은 평가** — val macro F1 최고 시점 체크포인트로 test 를 채점한다.

따라서 결과 차이는 conv 종류의 차이로 읽을 수 있다.

주의: 2번은 처음에 전역 시드만 되돌려 깨져 있었다. 증강 난수가 Dataset 안에 살아
있어 모델 간에 이어졌고, 같은 설정의 GAT 이 0.8332 와 0.7740 으로 갈렸다.

평가는 두 가지 방식을 지원한다.

- **단일 분할** (`run_benchmark`) — 빠르지만 test 가 작아 희소 클래스 추정이 불안정하다.
- **k-fold 교차검증** (`run_cv`) — 권장. fold 를 이어붙이면 전체 25,519장이 정확히
  한 번씩 채점되므로 Near-full 도 22장이 아니라 149장으로 평가된다.
  표본 잡음이 +-0.010 에서 +-0.004 로 줄고, test 가 val 과 분리돼 편향도 사라진다.

앙상블 조합은 `select_on="val"` 이 기본이다. test 로 조합을 고르면 그 점수는
낙관 편향돼 보고할 수 없다.

사용 예)
    python src/train_graph.py --data wafer_graph.npz --epochs 100
    python src/train_graph.py --data wafer_graph.npz --seeds 42 43 44   # 평균 +- 표준편차
    python src/train_graph.py --data wafer_graph.npz --cv 5             # 5-fold CV
    python src/train_graph.py --data wafer_graph.npz --split lot --holdout-test
"""


import argparse
import os
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# 노트북 생성기는 **최상단** import 만 제거해 인라인 네임스페이스로 대체한다.
# 따라서 이 모듈이 wafer_dataset 에서 쓰는 이름은 전부 여기에 모아야 한다.
# 함수 안에서 늦게 import 하면 노트북에서 ModuleNotFoundError 가 난다.

NUM_CLASSES = 8


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def model_forward(model, batch):
    """모델 종류에 무관하게 같은 방식으로 호출한다.

    그래프 전용 모델은 (x, edge_index, batch, scalars) 를 받고,
    이미지 트랙을 쓰는 모델(CNN 단독·융합)은 polar 를 하나 더 받는다.
    `needs_polar` 속성으로 구분해 학습 루프를 한 벌만 유지한다.
    """
    if getattr(model, "needs_polar", False):
        if not hasattr(batch, "polar"):
            raise ValueError(
                "이 모델은 극좌표 이미지를 요구하는데 배치에 polar 가 없다. "
                "build_dataset(..., include_polar=True) 로 만든 npz 를 쓸 것.")
        return model(batch.x, batch.edge_index, batch.batch, batch.scalars, batch.polar)
    return model(batch.x, batch.edge_index, batch.batch, batch.scalars)


@torch.no_grad()
def evaluate(model, loader, device, criterion=None, return_probs=False):
    """예측·정답·평균손실을 돌려준다. return_probs=True 면 softmax 확률도 함께.

    확률은 앙상블(soft voting)에 쓴다. argmax 만 저장하면 나중에 모델을 합칠 때
    확신도를 알 수 없어 hard voting 밖에 못 한다.
    """
    model.eval()
    preds, labels, probs, total = [], [], [], 0.0
    for batch in loader:
        batch = batch.to(device)
        out = model_forward(model, batch)
        if criterion is not None:
            total += criterion(out, batch.y).item() * batch.num_graphs
        preds.append(out.argmax(1).cpu())
        labels.append(batch.y.cpu())
        if return_probs:
            probs.append(torch.softmax(out, dim=1).cpu())
    loss = total / max(len(loader.dataset), 1)
    if return_probs:
        return (torch.cat(preds).numpy(), torch.cat(labels).numpy(), loss,
                torch.cat(probs).numpy().astype(np.float32))
    return torch.cat(preds).numpy(), torch.cat(labels).numpy(), loss


# ----------------------------------------------------------------------
# 결과 캐시. Colab 은 장시간 실행 중 끊기는 일이 잦은데, 네 모델을 1~2시간 돌리는
# 도중에 끊기면 전부 날아간다. 모델 하나가 끝날 때마다 저장해 두고, 다시 실행하면
# 이미 끝난 조합은 건너뛴다.
# ----------------------------------------------------------------------
_SCALAR_KEYS = ("conv", "name", "params", "best_val_f1", "test_macro_f1",
                "test_acc", "seconds", "seed", "best_epoch", "stopped_at", "jk",
                "fold")
# val_probs / val_labels 는 앙상블 조합을 **val 에서** 고르기 위한 것이다.
# test 확률로 조합을 고르면 그 점수는 낙관 편향돼 최종 보고에 쓸 수 없다.
_ARRAY_KEYS = ("per_class_f1", "preds", "labels", "confusion", "probs",
               "val_probs", "val_labels")


def cache_path(cache_dir, conv, seed, tag="", fold=None):
    f = "" if fold is None else f"_fold{fold}"
    return os.path.join(cache_dir, f"result_{tag}{conv}{f}_seed{seed}.npz")


def save_result(r, path):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    np.savez(path, **{k: np.asarray(r[k]) for k in _SCALAR_KEYS + _ARRAY_KEYS
                      if r.get(k) is not None})


def load_result(path):
    """예전 캐시에는 없는 키가 있을 수 있으므로 있는 것만 읽는다."""
    z = np.load(path, allow_pickle=False)
    r = {k: z[k].item() for k in _SCALAR_KEYS if k in z.files}
    r.update({k: z[k] for k in _ARRAY_KEYS if k in z.files})
    return r


def train_one(conv, loaders, packed, epochs=50, lr=1e-4, weight_decay=1e-5,
              patience=10, hidden=128, num_layers=3, heads=4, knn_k=8,
              seed=42, device=None, verbose=True, save=None, use_class_weights=True,
              jk=None, label=None, weight_alpha=1.0, model=None):
    """conv 하나를 학습하고 test 성적을 돌려준다.

    기본 하이퍼파라미터는 Seo & Kang (2026) 설정을 따른다.
    lr 1e-4, weight decay 1e-5, val macro F1 기준 early stopping(patience 10).
    v1 은 lr 1e-3 에 weight decay 도 early stopping 도 없어서 val loss 가
    0.483 -> 2.662 로 폭증했다.
    """
    train_loader, val_loader, test_loader = loaders
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 학습을 시작하기 전에 무작위성을 전부 초기 상태로 되돌린다.
    #
    # set_seed 만으로는 부족하다. 증강 난수는 Dataset 객체 안에 살아 있어서,
    # 로더 하나를 여러 모델이 공유하면 스트림이 이어진다. 첫 모델이 5000개를 쓰면
    # 두 번째 모델은 5001번부터 시작해 서로 다른 증강 데이터를 보게 된다.
    # 실제로 이 때문에 같은 설정의 GAT 이 0.8332 와 0.7740 으로 갈렸다.
    #
    # 로더 generator 도 되돌린다. 이건 셔플 순서와 워커 시드를 정하는데, 전역 난수를
    # 쓰면 파라미터 수가 다른 모델이 초기화에서 난수를 다르게 소비해 순서가 어긋난다.
    set_seed(seed)
    for _loader in loaders:
        ds = getattr(_loader, "dataset", None)
        if hasattr(ds, "reset_rng"):
            ds.reset_rng()
    gen = getattr(train_loader, "generator", None)
    if gen is not None:
        gen.manual_seed(getattr(train_loader, "_seed", seed))

    # model 을 직접 주면 그것을 쓴다(CNN 단독·융합처럼 GraphClassifier 가 아닌 경우).
    if model is None:
        model = GraphClassifier(conv=conv, hidden=hidden, num_layers=num_layers,
                                heads=heads, knn_k=knn_k, jk=jk)
    model = model.to(device)
    n_params = count_parameters(model)

    # 균형 샘플러를 쓸 때는 가중치를 꺼야 한다. 둘 다 걸면 소수 클래스가
    # 이중 증폭돼 recall 은 오르지만 precision 이 무너져 F1 이 오히려 떨어진다.
    if use_class_weights:
        weights = class_weights(packed.labels[train_loader.dataset.indices], NUM_CLASSES,
                                alpha=weight_alpha)
        criterion = nn.CrossEntropyLoss(
            weight=torch.tensor(weights, dtype=torch.float32, device=device))
    else:
        criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_f1, best_state, best_epoch = 0.0, None, 0
    stopped_at = epochs
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model_forward(model, batch)
            loss = criterion(out, batch.y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * batch.num_graphs
        train_loss /= len(train_loader.dataset)

        preds, labels, val_loss = evaluate(model, val_loader, device, criterion)
        val_f1 = f1_score(labels, preds, average="macro")
        mark = ""
        if val_f1 > best_f1:
            best_f1, best_epoch = val_f1, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            mark = " *"
        if verbose and (epoch % 10 == 0 or mark or epoch == epochs):
            print(f"  [{DISPLAY_NAME[conv]:<12}] Epoch {epoch:>3}/{epochs} | "
                  f"Train {train_loss:.4f} | Val {val_loss:.4f} | Val F1 {val_f1:.4f}{mark}",
                  flush=True)

        # early stopping: patience 회 연속 개선이 없으면 중단한다.
        if patience and epoch - best_epoch >= patience:
            stopped_at = epoch
            if verbose:
                print(f"  [{DISPLAY_NAME[conv]:<12}] {patience} epoch 동안 개선 없음 "
                      f"-> epoch {epoch} 에서 조기 종료 (최고는 epoch {best_epoch})", flush=True)
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    if save:
        torch.save(model.state_dict(), save)

    preds, labels, _, probs = evaluate(model, test_loader, device, return_probs=True)
    # val 확률도 남긴다. 앙상블 조합을 val 에서 골라야 test 점수가 편향되지 않는다.
    v_preds, v_labels, _, v_probs = evaluate(model, val_loader, device, return_probs=True)
    per_class = f1_score(labels, preds, average=None,
                         labels=list(range(NUM_CLASSES)), zero_division=0)
    return {
        "conv": conv,
        "name": label or DISPLAY_NAME[conv],
        "jk": str(jk),
        "params": n_params,
        "best_val_f1": best_f1,
        "best_epoch": best_epoch,
        "stopped_at": stopped_at,
        "test_macro_f1": f1_score(labels, preds, average="macro"),
        "test_acc": (preds == labels).mean(),
        "per_class_f1": per_class,
        "preds": preds,
        "labels": labels,
        "probs": probs,
        "val_preds": v_preds,
        "val_labels": v_labels,
        "val_probs": v_probs,
        "confusion": confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES))),
        "seconds": time.time() - t0,
    }


def run_benchmark(npz_path, convs=CONV_NAMES, seeds=(42,), epochs=50, batch_size=64,
                  lr=1e-4, weight_decay=1e-5, patience=10, hidden=128, num_layers=3,
                  heads=4, knn_k=8, augment=True, split="stratified", holdout_test=False,
                  num_workers=2, verbose=True, cache_dir=None, tag="",
                  balanced_sampler=False, use_class_weights=True, jk=None,
                  split_seed=None):
    """네 모델을 같은 조건으로 학습하고 결과 목록을 돌려준다.

    cache_dir 를 주면 모델 하나가 끝날 때마다 결과를 저장하고, 다시 실행할 때
    이미 끝난 조합은 건너뛴다. Colab 이 끊겨도 이어서 돌릴 수 있다.
    tag 는 서로 다른 설정(예: lot 분할)의 캐시를 구분하는 접두사다.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"장치: {device} | 모델 {list(convs)} | seed {list(seeds)} | epoch {epochs}\n")

    results = []
    for seed in seeds:
        # 분할은 seed 당 한 번만. 네 모델이 완전히 같은 데이터를 본다.
        tr, va, te, packed = make_loaders(
            npz_path, batch_size=batch_size, augment=augment, split=split,
            holdout_test=holdout_test, seed=seed, num_workers=num_workers,
            balanced_sampler=balanced_sampler, split_seed=split_seed)
        for conv in convs:
            print(f"\n--- seed {seed} / {DISPLAY_NAME[conv]} ---")
            cp = cache_path(cache_dir, conv, seed, tag) if cache_dir else None
            if cp and os.path.exists(cp):
                r = load_result(cp)
                results.append(r)
                print(f"  (캐시에서 불러옴) test macro F1 {r['test_macro_f1']:.4f}")
                continue
            r = train_one(conv, (tr, va, te), packed, epochs=epochs, lr=lr,
                          weight_decay=weight_decay, patience=patience, hidden=hidden,
                          num_layers=num_layers, heads=heads, knn_k=knn_k,
                          seed=seed, device=device, verbose=verbose,
                          use_class_weights=use_class_weights, jk=jk)
            r["seed"] = seed
            if cp:
                save_result(r, cp)
            results.append(r)
            print(f"  -> test macro F1 {r['test_macro_f1']:.4f} "
                  f"({r['seconds'] / 60:.1f}분, 파라미터 {r['params']:,})")
    return results


# ----------------------------------------------------------------------
# k-fold 교차검증
#
# 단일 분할의 문제는 test 가 작다는 것이다. Near-full 은 전체 149장 중 test 에
# 22장뿐이라 한 장만 틀려도 recall 이 0.045 움직이고, 이 표본 잡음만으로
# macro F1 이 +-0.010 흔들린다. 시드를 늘려도 같은 작은 시험지를 다시 채점할
# 뿐이라 줄지 않는다.
#
# CV 는 fold 를 이어붙이면 25,519장 전부가 정확히 한 번씩 채점되므로 이 잡음이
# 사실상 사라진다(+-0.004). 편향도 함께 없어진다 -- test 가 val 과 분리된다.
# ----------------------------------------------------------------------
def run_cv(npz_path, configs, n_folds=5, split_seed=42, seed=42, epochs=50,
           batch_size=64, lr=1e-4, weight_decay=1e-5, patience=10, hidden=128,
           num_layers=3, heads=4, knn_k=8, augment=True, split="stratified",
           num_workers=2, verbose=True, cache_dir=None, folds=None):
    """설정 여러 개를 k-fold 로 학습한다.

    Parameters
    ----------
    configs : {표시이름: dict(conv=..., balanced_sampler=..., use_class_weights=..., jk=...)}
        예) {"GIN[base]": dict(conv="gin", balanced_sampler=False,
                               use_class_weights=True, jk=None)}
    split_seed : 분할 시드. **모든 설정·모든 실행에서 고정**해야 fold 가 같아지고
        모델 비교가 짝지어진다. 학습 무작위성(seed)과 분리되어 있다.
    folds : 돌릴 fold 번호 목록. None 이면 0..n_folds-1 전부. Colab 세션을 나눠
        돌릴 때 [0,1] / [2,3,4] 처럼 쪼갤 수 있다.

    fold 하나마다 로더를 만들고, 그 fold 안에서 모든 설정을 학습한다. 같은 fold 의
    결과끼리는 test 집합이 같으므로 앙상블이 가능하다.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    folds = list(range(n_folds)) if folds is None else list(folds)
    print(f"장치: {device} | 설정 {list(configs)} | fold {folds}/{n_folds} "
          f"| split_seed {split_seed} | epoch {epochs}\n")

    results, packed = [], None
    for fold in folds:
        # 균형 샘플러가 필요한 설정이 있으면 로더를 두 벌 만든다. 분할은 동일하고
        # 배치를 뽑는 방식만 다르므로 test 집합이 같다(앙상블 전제).
        need = {bool(c.get("balanced_sampler", False)) for c in configs.values()}
        loaders = {}
        for bal in sorted(need):
            tr, va, te, packed = make_loaders(
                npz_path, batch_size=batch_size, augment=augment, split=split,
                seed=seed, split_seed=split_seed, fold=fold, n_folds=n_folds,
                num_workers=num_workers, balanced_sampler=bal, packed=packed)
            loaders[bal] = (tr, va, te)
        if len(loaders) == 2:
            assert np.array_equal(loaders[False][2].dataset.indices,
                                  loaders[True][2].dataset.indices), "test 집합이 다르다"

        for name, cfg in configs.items():
            print(f"\n--- fold {fold} / {name} ---")
            cp = (cache_path(cache_dir, cfg["conv"], seed, tag=f"{_slug(name)}_",
                             fold=fold)
                  if cache_dir else None)
            if cp and os.path.exists(cp):
                r = load_result(cp)
                results.append(r)
                print(f"  (캐시에서 불러옴) test macro F1 {r['test_macro_f1']:.4f}")
                continue
            r = train_one(cfg["conv"], loaders[bool(cfg.get("balanced_sampler", False))],
                          packed, epochs=epochs, lr=lr, weight_decay=weight_decay,
                          patience=patience, hidden=hidden, num_layers=num_layers,
                          heads=heads, knn_k=knn_k, seed=seed, device=device,
                          verbose=verbose, jk=cfg.get("jk"),
                          use_class_weights=cfg.get("use_class_weights", True),
                          label=name)
            r["seed"], r["fold"] = seed, fold
            if cp:
                save_result(r, cp)
            results.append(r)
            print(f"  -> test macro F1 {r['test_macro_f1']:.4f} "
                  f"({r['seconds'] / 60:.1f}분, 파라미터 {r['params']:,})")
    return results


def _slug(name):
    """설정 이름을 파일명에 쓸 수 있게 정리한다."""
    return "".join(c if c.isalnum() else "-" for c in name).strip("-")


# ----------------------------------------------------------------------
# 사양서 §4.4 / §5.2 / §5.3 — 봉인 분할 위에서 도는 개발·최종 평가
# ----------------------------------------------------------------------
def spec_cache_path(cache_dir, preproc_hash, split_mode, conv, config, fold, seed):
    """§4.4 캐시 키 = 전처리해시/분할모드/conv/설정/fold/시드.

    fold 가 빠지면 5개 fold 가 같은 파일을 덮어쓴다. 전처리 해시가 빠지면
    전처리를 바꾼 뒤에도 옛 결과를 그대로 불러온다.
    """
    name = f"{preproc_hash}_{split_mode}_{conv}_{_slug(config)}_fold{fold}_seed{seed}.npz"
    return os.path.join(cache_dir, name)


def run_dev_stage(packed, split, configs, preproc_hash, split_mode="sealed-grouped",
                  seed=42, epochs=50, batch_size=64, lr=1e-4, weight_decay=1e-5,
                  patience=10, hidden=128, num_layers=3, heads=4, knn_k=8,
                  augment=True, num_workers=2, verbose=True, cache_dir=None,
                  ckpt_dir=None, folds=None):
    """§3.1 개발 구간에서 fold 별로 설정들을 학습한다. 최종 평가 집합은 건드리지 않는다.

    ckpt_dir 를 주면 fold 별 최고 체크포인트를 저장한다. §5.3 최종 평가가
    이 체크포인트들의 소프트 보팅으로 수행되므로 **저장하지 않으면 최종 평가를
    할 수 없다**(재학습 금지).

    돌려주는 결과의 test_* 필드는 **검증 집합** 성적이다. §3.3 대로 조기 종료
    편향을 포함하므로 상대 비교에만 쓴다.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    folds = list(range(len(split["folds"]))) if folds is None else list(folds)
    print(f"장치: {device} | 설정 {list(configs)} | fold {folds} | epoch {epochs}\n")

    results = []
    for fold in folds:
        tr_idx, va_idx = split["folds"][fold]
        need = {bool(c.get("balanced_sampler", False)) for c in configs.values()}
        loaders = {b: loaders_from_indices(
            packed, tr_idx, va_idx, batch_size=batch_size, augment=augment, seed=seed,
            num_workers=num_workers, balanced_sampler=b, verbose=(b == min(need)))
            for b in sorted(need)}

        for name, cfg in configs.items():
            print(f"\n--- fold {fold} / {name} ---")
            cp = (spec_cache_path(cache_dir, preproc_hash, split_mode, cfg["conv"],
                                  name, fold, seed) if cache_dir else None)
            ck = (os.path.join(ckpt_dir, f"{_slug(name)}_fold{fold}_seed{seed}.pt")
                  if ckpt_dir else None)
            if cp and os.path.exists(cp) and (ck is None or os.path.exists(ck)):
                r = load_result(cp)
                results.append(r)
                print(f"  (캐시) 검증 macro F1 {r['test_macro_f1']:.4f}")
                continue
            # build 를 주면 그 모델을 쓴다(CNN 단독·융합처럼 GraphClassifier 가 아닌 경우).
            # 매 fold 새로 만들어야 이전 fold 의 가중치가 이어지지 않는다.
            built = cfg["build"]() if callable(cfg.get("build")) else None
            r = train_one(cfg.get("conv", "gin"),
                          loaders[bool(cfg.get("balanced_sampler", False))],
                          packed, epochs=epochs, lr=lr, weight_decay=weight_decay,
                          patience=patience, hidden=hidden, num_layers=num_layers,
                          heads=heads, knn_k=knn_k, seed=seed, device=device,
                          verbose=verbose, jk=cfg.get("jk"), save=ck,
                          use_class_weights=cfg.get("use_class_weights", True),
                          weight_alpha=cfg.get("weight_alpha", 1.0),
                          model=built, label=name)
            r["seed"], r["fold"] = seed, fold
            if cp:
                save_result(r, cp)
            results.append(r)
            print(f"  -> 검증 macro F1 {r['test_macro_f1']:.4f} ({r['seconds'] / 60:.1f}분)")
    return results


def paired_comparison(results, name_a, name_b, metric="test_macro_f1"):
    """§5.2 fold 단위 대응 비교.

    같은 fold 의 두 설정을 짝지어 차분한다. 분할에서 유래하는 변동이 상쇄되므로
    독립 평균 비교보다 검출력이 높다. b - a 를 본다(a 가 기준선).

    주의: fold 가 5개면 부호검정의 최소 p 값이 1/2^5 = 0.031(단측)이다.
    '개선 5/5 fold' 가 이 설계로 낼 수 있는 가장 강한 주장이며 그 이상은 불가능하다.
    """
    by = {}
    for r in results:
        if r["name"] in (name_a, name_b) and r.get("fold", -1) >= 0:
            by.setdefault(r["fold"], {})[r["name"]] = r[metric]
    folds = sorted(f for f, d in by.items() if name_a in d and name_b in d)
    if not folds:
        raise ValueError(f"'{name_a}' 와 '{name_b}' 를 짝지을 fold 가 없다")

    diffs = np.array([by[f][name_b] - by[f][name_a] for f in folds], dtype=float)
    sd = float(diffs.std(ddof=1)) if len(diffs) > 1 else 0.0
    n_up = int((diffs > 0).sum())
    return {
        "기준선": name_a, "비교": name_b, "fold수": len(folds),
        "평균차": float(diffs.mean()), "표준편차": sd,
        "개선fold": n_up, "fold별차": diffs,
        "표기": f"{diffs.mean():+.4f} ± {sd:.4f} (개선 {n_up}/{len(folds)} fold)",
    }


def paired_table(results, baseline, metric="test_macro_f1"):
    """기준선 대비 모든 설정의 대응 비교표."""
    import pandas as pd

    names = sorted({r["name"] for r in results if r.get("fold", -1) >= 0})
    rows = []
    for n in names:
        if n == baseline:
            continue
        try:
            rows.append(paired_comparison(results, baseline, n, metric))
        except ValueError:
            continue
    return pd.DataFrame([{k: v for k, v in r.items() if k != "fold별차"} for r in rows])


@torch.no_grad()
def final_evaluation(packed, split, ckpt_specs, batch_size=64, num_workers=2,
                     hidden=128, num_layers=3, heads=4, knn_k=8, device=None):
    """§5.3 봉인된 최종 평가 집합에서 **1회만** 수행한다.

    개발 단계에서 저장한 fold 별 체크포인트의 소프트 보팅이며 재학습하지 않는다.

    ckpt_specs : [(conv, jk, 체크포인트경로), ...]
        여기 넣는 조합은 개발 구간에서 **이미 확정**되어 있어야 한다. 결과를 보고
        조합을 바꾸면 이 집합은 검증 집합으로 강등된다(§0.4, §5.3).
    """
    from torch_geometric.loader import DataLoader

    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    final_idx = split["final"]
    loader = DataLoader(WaferDualTrackDataset(packed, final_idx, augment=False),
                        batch_size=batch_size, shuffle=False, num_workers=num_workers)

    print(f"[§5.3] 최종 평가 — 봉인 집합 {len(final_idx):,}장, "
          f"체크포인트 {len(ckpt_specs)}개 소프트 보팅 (재학습 없음)")

    prob_sum, labels = None, None
    for conv, jk, path in ckpt_specs:
        model = GraphClassifier(conv=conv, hidden=hidden, num_layers=num_layers,
                                heads=heads, knn_k=knn_k, jk=jk).to(device)
        model.load_state_dict(torch.load(path, map_location=device))
        preds, labels, _, probs = evaluate(model, loader, device, return_probs=True)
        prob_sum = probs if prob_sum is None else prob_sum + probs
        print(f"   {os.path.basename(path)}  단독 macro F1 "
              f"{f1_score(labels, preds, average='macro'):.4f}")

    probs = prob_sum / len(ckpt_specs)
    preds = probs.argmax(1)
    return {
        "conv": "final", "name": "최종(fold 소프트보팅)", "stage": "final",
        "n": len(final_idx), "params": -1, "fold": -1, "seed": -1,
        "best_val_f1": float("nan"), "best_epoch": -1, "stopped_at": -1, "seconds": 0.0,
        "test_macro_f1": f1_score(labels, preds, average="macro"),
        "test_acc": (preds == labels).mean(),
        "balanced_acc": _balanced_accuracy(labels, preds),
        "per_class_f1": f1_score(labels, preds, average=None,
                                 labels=list(range(NUM_CLASSES)), zero_division=0),
        "support": np.bincount(labels, minlength=NUM_CLASSES),
        "preds": preds, "labels": labels, "probs": probs,
        "confusion": confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES))),
    }


def _balanced_accuracy(labels, preds):
    """§5.1 보조 지표. 클래스별 재현율의 평균."""
    from sklearn.metrics import balanced_accuracy_score
    return float(balanced_accuracy_score(labels, preds))


FOLD_SD_FOOTNOTE = ("fold 간 학습 집합이 상호 중첩되므로 이 편차는 "
                    "독립 표본의 신뢰구간이 아니다.")


# ----------------------------------------------------------------------
# 1 fold 스크리닝용 진단
#
# fold 별 차분의 표준편차는 실측 약 0.02 다(0.0188 / 0.0220 / 0.0245 / 0.0257).
# 따라서 1 fold 로 부호를 틀릴 확률은 효과 크기에 따라 이렇다.
#     효과 0.005 -> 40%   0.02 -> 16%   0.04 -> 2%   0.06 -> 0.1%
# 0.04 보다 큰 효과는 1 fold 로 판별되고, 그보다 작으면 5 fold 가 필요하다.
# ----------------------------------------------------------------------
SCREEN_SD = 0.02          # fold 간 차분 표준편차 실측치
SCREEN_RESOLVABLE = 0.04  # 1 fold 로 판별 가능한 최소 효과


def screen_verdict(delta, sd=SCREEN_SD):
    """1 fold 차분 하나를 보고 다음 행동을 정한다."""
    if delta >= 3 * sd:
        return "채택 유력", "5 fold 는 최종 후보일 때만"
    if delta >= 2 * sd:
        return "유망", "5 fold 로 확정할 것"
    if delta > -2 * sd:
        return "판정 불가", f"효과가 1 fold 해상도({2 * sd:+.3f}) 밖. 5 fold 필요"
    return "버림", "5 fold 돌릴 가치 없음"


def oracle_upper_bound(res_a, res_b, class_names=None):
    """두 모델을 웨이퍼마다 맞은 쪽으로 고르는 가상의 완벽한 심판.

    **어떤 융합·앙상블도 이 값을 넘을 수 없다.** 융합을 만들기 전에 이걸 재면
    천장이 얼마인지 알 수 있어, 가망 없는 방향에 시간을 쓰지 않는다.

    참고: 이전에 GIN+GAT 오라클은 단독 대비 +0.01 수준이었고 실제 앙상블 이득도
    +0.021 에 그쳤다. 두 그래프 모델이 너무 비슷했기 때문이다. 표현 방식이 다른
    CNN 은 더 클 수 있으나, 재보기 전에는 알 수 없다.
    """
    la, lb = res_a["labels"], res_b["labels"]
    if not np.array_equal(la, lb):
        raise ValueError("두 결과의 평가 집합이 다르다 - 같은 fold 인지 확인할 것")
    pa, pb, y = res_a["preds"], res_b["preds"], la

    ok_a, ok_b = pa == y, pb == y
    oracle = np.where(ok_a, pa, pb)

    f1_a = f1_score(y, pa, average="macro")
    f1_b = f1_score(y, pb, average="macro")
    f1_o = f1_score(y, oracle, average="macro")
    best = max(f1_a, f1_b)

    out = {
        "a": res_a["name"], "b": res_b["name"],
        "macro_a": f1_a, "macro_b": f1_b, "oracle": f1_o,
        "여지": f1_o - best,
        "둘다맞음": float((ok_a & ok_b).mean()),
        "A만맞음": float((ok_a & ~ok_b).mean()),
        "B만맞음": float((~ok_a & ok_b).mean()),
        "둘다틀림": float((~ok_a & ~ok_b).mean()),
    }
    gap = out["여지"]
    if gap >= 0.05:
        out["판정"] = "융합 진행 — 여지가 충분하다"
    elif gap >= 0.02:
        out["판정"] = "애매 — 시간 여유가 있으면"
    else:
        out["판정"] = "융합 포기 — 천장이 낮아 어떻게 만들어도 안 된다"

    if class_names is not None:
        rows = []
        for c, nm in enumerate(class_names):
            m = y == c
            if not m.any():
                continue
            rows.append({"패턴": nm, "지원수": int(m.sum()),
                         "A정답률": float(ok_a[m].mean()),
                         "B정답률": float(ok_b[m].mean()),
                         "B만맞음": float((~ok_a & ok_b)[m].mean())})
        out["클래스별"] = rows
    return out


def dev_summary(results, metric="test_macro_f1"):
    """§5.4 개발 지표 = 설정별 `평균 ± fold 표준편차`."""
    import pandas as pd

    by = {}
    for r in results:
        if r.get("fold", -1) >= 0:
            by.setdefault(r["name"], []).append(r[metric])
    rows = []
    for name, vals in by.items():
        v = np.asarray(vals, dtype=float)
        rows.append({"설정": name, "fold수": len(v), "평균": v.mean(),
                     "fold_표준편차": v.std(ddof=1) if len(v) > 1 else 0.0,
                     "표기": f"{v.mean():.4f} ± {v.std(ddof=1) if len(v) > 1 else 0.0:.4f}"})
    return pd.DataFrame(rows).sort_values("평균", ascending=False).set_index("설정")


def per_class_with_support(result, class_names):
    """§5.1 클래스별 F1 + **지원 수 병기**.

    지원 수를 빼면 Near-full(최종 평가 45장 내외)의 F1 변동을 Edge-Ring(2,900장
    내외)과 같은 무게로 읽게 된다. 사양서가 병기를 필수로 둔 이유다.
    """
    import pandas as pd

    sup = result.get("support")
    if sup is None:
        sup = np.bincount(result["labels"], minlength=NUM_CLASSES)
    f1 = result["per_class_f1"]
    df = pd.DataFrame({"F1": f1, "지원수": sup}, index=class_names)
    df["1장당_영향"] = np.where(sup > 0, 1.0 / np.maximum(sup, 1), np.nan)
    return df.round(4)


def pool_folds(results):
    """fold 결과를 이어붙여 '모든 웨이퍼를 한 번씩 채점한' 성적을 만든다.

    fold 평균이 아니라 **이어붙이기(pooling)** 다. 평균은 fold 마다 클래스 개수가
    조금씩 달라 희소 클래스에서 흔들리지만, 이어붙이면 Near-full 을 149장 전부로
    한 번에 채점하므로 훨씬 안정적이다. 이 값을 논문 표에 쓴다.
    """
    by = {}
    for r in results:
        if r.get("fold", -1) >= 0:
            by.setdefault(r["name"], []).append(r)
    out = {}
    for name, rs in by.items():
        rs = sorted(rs, key=lambda r: r.get("fold", 0))
        preds = np.concatenate([r["preds"] for r in rs])
        labels = np.concatenate([r["labels"] for r in rs])
        out[name] = {
            "conv": rs[0]["conv"], "name": name, "n_folds": len(rs),
            "params": rs[0]["params"], "n": len(labels),
            "test_macro_f1": f1_score(labels, preds, average="macro"),
            "test_acc": (preds == labels).mean(),
            "per_class_f1": f1_score(labels, preds, average=None,
                                     labels=list(range(NUM_CLASSES)), zero_division=0),
            "fold_macro_f1": np.array([r["test_macro_f1"] for r in rs]),
            "preds": preds, "labels": labels,
            "confusion": confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES))),
            "seconds": sum(r["seconds"] for r in rs),
        }
    return out


def summarize_cv(results):
    """CV 결과를 (fold별 표, 요약표) 로 만든다.

    요약표의 `macro_F1_pooled` 가 보고할 값이고, `fold_std` 는 안정성 지표다.
    둘을 헷갈리면 안 된다 -- pooled 는 점추정치, fold_std 는 그 주변의 흔들림이다.
    """
    import pandas as pd

    per_fold = pd.DataFrame([{
        "설정": r["name"], "fold": r.get("fold", -1), "seed": r.get("seed", -1),
        "macro_F1": r["test_macro_f1"], "acc": r["test_acc"],
        "best_val_F1": r["best_val_f1"], "최고epoch": r.get("best_epoch", -1),
        "분": r["seconds"] / 60,
    } for r in results]).sort_values(["설정", "fold"])

    pooled = pool_folds(results)
    rows = []
    for name, p in pooled.items():
        f = p["fold_macro_f1"]
        rows.append({
            "설정": name, "파라미터": p["params"], "fold수": p["n_folds"],
            "채점표본": p["n"],
            "macro_F1_pooled": p["test_macro_f1"],
            "fold_평균": f.mean(), "fold_std": f.std(ddof=1) if len(f) > 1 else 0.0,
            "acc": p["test_acc"], "총분": p["seconds"] / 60,
        })
    agg = (pd.DataFrame(rows).set_index("설정")
           .sort_values("macro_F1_pooled", ascending=False))
    return per_fold, agg


def cv_per_class_table(results):
    """설정별 x 패턴별 F1 (pooling 기준)."""
    import pandas as pd

    names = [REVERSE_LABEL_MAP[i] for i in range(NUM_CLASSES)]
    pooled = pool_folds(results)
    return pd.DataFrame({k: v["per_class_f1"] for k, v in pooled.items()},
                        index=names).T.round(4)


def summarize(results):
    """결과 목록을 비교표(DataFrame)로 만든다. seed 가 여럿이면 평균 +- 표준편차."""
    import pandas as pd

    df = pd.DataFrame([{
        "모델": r["name"], "seed": r["seed"], "파라미터": r["params"],
        "test_macro_F1": r["test_macro_f1"], "test_acc": r["test_acc"],
        "best_val_F1": r["best_val_f1"], "최고epoch": r.get("best_epoch", -1),
        "종료epoch": r.get("stopped_at", -1), "분": r["seconds"] / 60,
    } for r in results])

    if df["seed"].nunique() > 1:
        agg = df.groupby("모델").agg(
            파라미터=("파라미터", "first"),
            macro_F1_평균=("test_macro_F1", "mean"),
            macro_F1_표준편차=("test_macro_F1", "std"),
            acc_평균=("test_acc", "mean"),
            분=("분", "mean"),
        ).sort_values("macro_F1_평균", ascending=False)
    else:
        agg = df.set_index("모델")[["파라미터", "test_macro_F1", "test_acc",
                                   "best_val_F1", "최고epoch", "종료epoch", "분"]] \
                .sort_values("test_macro_F1", ascending=False)
    return df, agg


def per_class_table(results):
    """패턴별 F1 비교표. 어떤 모델이 어떤 패턴에 강한지 본다."""
    import pandas as pd

    names = [REVERSE_LABEL_MAP[i] for i in range(NUM_CLASSES)]
    rows = {}
    for r in results:
        rows.setdefault(r["name"], []).append(r["per_class_f1"])
    data = {k: np.mean(v, axis=0) for k, v in rows.items()}
    return pd.DataFrame(data, index=names).T.round(4)






# ----------------------------------------------------------------------
# 앙상블 (soft voting)
# ----------------------------------------------------------------------
def ensemble(results, convs=None, weights=None):
    """여러 모델의 softmax 확률을 평균내 하나의 예측을 만든다.

    모델마다 잘하는 패턴이 다르다는 점을 이용한다. 실측에서 GIN 은 Center/Loc 에,
    GCN 은 Random 에, NoMsg 는 Scratch 에 강했다. 확률을 평균내면 각자의 강점이
    부분적으로 합쳐진다.

    argmax 가 아니라 확률을 평균내는 것이 중요하다(soft voting). 확신이 강한 모델의
    의견이 더 반영된다.

    Parameters
    ----------
    results : train_one() 결과 목록. 모두 같은 test 집합이어야 한다.
    convs : 사용할 모델 이름 목록. None 이면 전부.
    weights : 모델별 가중치. None 이면 균등.
    """
    rs = [r for r in results if convs is None or r["conv"] in convs]
    if not rs:
        raise ValueError("앙상블할 모델이 없다")
    if "probs" not in rs[0]:
        raise ValueError("확률이 저장돼 있지 않다. 모델을 다시 학습해야 한다"
                         " (예전 캐시는 probs 가 없다)")

    labels = rs[0]["labels"]
    for r in rs[1:]:
        if not np.array_equal(r["labels"], labels):
            raise ValueError("모델들의 test 집합이 다르다 - 같은 분할로 학습했는지 확인할 것")

    w = np.ones(len(rs)) if weights is None else np.asarray(weights, dtype=float)
    w = w / w.sum()
    probs = sum(wi * r["probs"] for wi, r in zip(w, rs))
    preds = probs.argmax(1)

    # val 에서의 앙상블 성적. 조합 선택은 **이 값**으로 해야 test 가 오염되지 않는다.
    val_macro, val_probs, v_labels = float("nan"), None, None
    if all(r.get("val_probs") is not None for r in rs):
        v_labels = rs[0]["val_labels"]
        if all(np.array_equal(r["val_labels"], v_labels) for r in rs):
            val_probs = sum(wi * r["val_probs"] for wi, r in zip(w, rs))
            val_macro = f1_score(v_labels, val_probs.argmax(1), average="macro")

    return {
        "conv": "ensemble",
        "name": "앙상블(" + " + ".join(r["name"] for r in rs) + ")",
        "params": sum(r["params"] for r in rs),
        "best_val_f1": val_macro,
        "val_macro_f1": val_macro,
        "best_epoch": -1, "stopped_at": -1, "seed": rs[0].get("seed", -1),
        "fold": rs[0].get("fold", -1),
        "test_macro_f1": f1_score(labels, preds, average="macro"),
        "test_acc": (preds == labels).mean(),
        "per_class_f1": f1_score(labels, preds, average=None,
                                 labels=list(range(NUM_CLASSES)), zero_division=0),
        "members": [r["name"] for r in rs],
        "preds": preds, "labels": labels, "probs": probs,
        "val_labels": v_labels, "val_probs": val_probs,
        "confusion": confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES))),
        "seconds": 0.0,
    }


def best_ensemble(results, max_models=4, select_on="val"):
    """조합을 탐색해 가장 좋은 앙상블을 찾는다.

    select_on="val"  (기본) — **val** 성적으로 조합을 고른다. test 는 손대지 않으므로
        1등 조합의 test 점수를 그대로 보고할 수 있다.
    select_on="test" — test 성적으로 고른다. 낙관 편향되므로 보고용으로 쓰면 안 되고,
        '앙상블 상한이 얼마나 되는가'를 볼 때만 쓴다.

    돌려주는 각 행은 (선택기준점수, 멤버이름목록, 앙상블결과) 이고 선택기준으로 정렬된다.
    앙상블결과에는 test 성적이 그대로 들어 있다.
    """
    from itertools import combinations

    pool = [r for r in results if r.get("probs") is not None and r["conv"] != "ensemble"]
    folds = {r.get("fold", -1) for r in pool}
    if len(folds) > 1:
        raise ValueError(
            f"결과에 fold 가 여러 개 섞여 있다({sorted(folds)}). fold 마다 test 집합이 "
            "다르므로 확률을 그대로 섞을 수 없다. CV 결과에는 best_cv_ensemble() 을 쓸 것.")
    if select_on == "val" and not all(r.get("val_probs") is not None for r in pool):
        print("[경고] val 확률이 없는 결과가 있어 test 기준으로 고른다 "
              "(예전 캐시는 val_probs 가 없다). 이 점수는 보고용이 아니다.")
        select_on = "test"

    rows = []
    for n in range(2, min(max_models, len(pool)) + 1):
        for combo in combinations(pool, n):
            e = ensemble(list(combo))
            key = e["val_macro_f1"] if select_on == "val" else e["test_macro_f1"]
            rows.append((key, [r["name"] for r in combo], e))
    rows.sort(key=lambda t: -t[0])
    return rows


def cv_ensemble(results, member_names):
    """fold 마다 지정한 설정들을 앙상블한다. 결과는 fold 개수만큼 나온다.

    fold 를 넘어 확률을 섞으면 안 된다 -- test 집합이 다르기 때문이다.
    같은 fold 안에서만 합치고, 최종 성적은 pool_folds 로 이어붙여 낸다.
    """
    by_fold = {}
    for r in results:
        if r["name"] in member_names and r.get("fold", -1) >= 0:
            by_fold.setdefault(r["fold"], []).append(r)

    out = []
    label = "앙상블(" + " + ".join(member_names) + ")"
    for fold in sorted(by_fold):
        members = by_fold[fold]
        if len(members) != len(member_names):
            continue          # 아직 안 돌린 설정이 있는 fold 는 건너뛴다
        e = ensemble(members)
        e["name"], e["fold"] = label, fold
        out.append(e)
    return out


def best_cv_ensemble(results, max_models=3, select_on="val"):
    """CV 전체에서 가장 좋은 앙상블 조합을 찾는다.

    조합 선택은 **fold 평균 val macro F1** 으로 한다. test 는 마지막 채점에만 쓴다.
    돌려주는 각 행은 (선택기준점수, 멤버이름목록, pooled 결과dict, fold별 결과목록).
    """
    from itertools import combinations

    names = sorted({r["name"] for r in results if r.get("fold", -1) >= 0
                    and r["conv"] != "ensemble"})
    rows = []
    for n in range(2, min(max_models, len(names)) + 1):
        for combo in combinations(names, n):
            es = cv_ensemble(results, list(combo))
            if not es:
                continue
            if select_on == "val":
                key = float(np.mean([e["val_macro_f1"] for e in es]))
            else:
                key = float(np.mean([e["test_macro_f1"] for e in es]))
            if np.isnan(key):
                continue
            pooled = pool_folds(es)[es[0]["name"]]
            rows.append((key, list(combo), pooled, es))
    rows.sort(key=lambda t: -t[0])
    return rows


# 9. 전처리 — 극좌표 포함본 생성

①③단계는 극좌표 이미지가 필요하다. 기존 `wafer_spec.npz` 는 그래프 전용이라
`polar` 가 없다.

**중요**: `dup_ids` 와 배제 기준을 동일하게 유지해야 기존 분할 파일을 그대로 쓸 수 있다.
웨이퍼 순서와 개수가 같아야 인덱스가 맞기 때문이다. 아래 셀이 이를 검증한다.

전체 25,519장에 약 5~8분, 파일 약 740MB.


In [ ]:
if os.path.exists(DRIVE_POLAR):
    !cp "$DRIVE_POLAR" "$POLAR_NPZ"
    print("Drive 에서 복사 완료")
else:
    build_dataset(RAW_PKL, POLAR_NPZ, knn_k=8, noise_eps_dies=2.0,
                  noise_min_samples=3, include_polar=True,
                  min_valid_die=0, min_defect_die=0)
    !cp "$POLAR_NPZ" "$DRIVE_POLAR"
    print("Drive 백업 완료")

packed = PackedWaferData(POLAR_NPZ)
print(f"\n웨이퍼 {len(packed):,} / polar {packed.polar.shape if packed.polar is not None else None}")
assert packed.polar is not None, "polar 가 없다 - include_polar=True 로 다시 만들 것"

split, split_meta = load_splits(SPLIT_PATH)
print("분할 로드:", {k: v for k, v in split_meta.items() if k != 'preproc_hash'})

# 분할 파일이 가리키는 인덱스가 이 npz 에서도 같은 웨이퍼인지 확인한다.
# 배제 기준이나 순서가 달라졌으면 여기서 걸린다.
assert len(packed) == len(split['dev']) + len(split['final']), (
    f"웨이퍼 수 불일치: npz {len(packed)} vs 분할 {len(split['dev'])+len(split['final'])}")
n_dup = int((np.bincount(packed.dup_ids) > 1).sum())
print(f"중복 그룹 {n_dup} (기대 93)")
assert n_dup == 93, "dup_ids 가 다르다 - 전처리 설정이 기존과 다르다"
print("\n인덱스 정합성 확인 완료 - 기존 분할을 그대로 쓸 수 있다")


  2000/25519  (13s)
  4000/25519  (25s)
  6000/25519  (39s)
  8000/25519  (53s)
  10000/25519  (71s)
  12000/25519  (86s)
  14000/25519  (98s)
  16000/25519  (106s)
  18000/25519  (116s)
  20000/25519  (126s)
  22000/25519  (135s)
  24000/25519  (145s)

완료: 25519개 저장, 0개 제외, 182s
배제 사유별 (§1.3): no_label 0, no_valid_die 0, no_defect 0, too_small 0, degenerate 0
중복 그룹 (§1.4): 25,425개 / 여분 웨이퍼 94장
평균 노드 수 240.0, 평균 방향성 엣지 수 1920.4
극좌표 이미지: 포함
산출물 해시 (§2.8): aa2c3a97d76e
-> /content/wafer_spec_polar.npz
Drive 백업 완료

웨이퍼 25,519 / polar (25519, 3, 64, 64)
분할 로드: {'mode': 'sealed-grouped', 'seed': 42, 'dev_frac': 0.7, 'n_folds': 5}
중복 그룹 93 (기대 93)

인덱스 정합성 확인 완료 - 기존 분할을 그대로 쓸 수 있다


# 10. 공통 설정 — fold 0 고정

스크리닝은 **항상 같은 fold** 에서 한다. 다른 fold 끼리 비교하면 fold 난이도가
섞여 무의미해진다.


In [ ]:
SCREEN_FOLD = 0
SEED, BATCH_SIZE, EPOCHS = 42, 64, 50
LR, WD, PATIENCE = 1e-4, 1e-5, 10
HIDDEN, NUM_LAYERS, HEADS = 128, 3, 4
PREPROC_HASH = split_meta["preproc_hash"]
names = [REVERSE_LABEL_MAP[i] for i in range(8)]

EXT_CACHE = f'{SPEC_DIR}/ext_results'
EXT_CKPT  = f'{SPEC_DIR}/ext_checkpoints'
for p in (EXT_CACHE, EXT_CKPT):
    os.makedirs(p, exist_ok=True)

SCREEN = {}          # 스크리닝 결과 모음

def screen(name, cfg, fold=SCREEN_FOLD, cache_dir=EXT_CACHE):
    """설정 하나를 fold 하나로 학습한다."""
    global SCREEN
    rs = run_dev_stage(packed, split, {name: cfg}, PREPROC_HASH,
                       split_mode=split_meta["mode"], seed=SEED, epochs=EPOCHS,
                       batch_size=BATCH_SIZE, lr=LR, weight_decay=WD,
                       patience=PATIENCE, hidden=HIDDEN, num_layers=NUM_LAYERS,
                       heads=HEADS, num_workers=2, cache_dir=cache_dir,
                       ckpt_dir=EXT_CKPT, folds=[fold])
    SCREEN[name] = rs[0]
    return rs[0]

print(f"스크리닝 fold = {SCREEN_FOLD} (고정)")
print(f"판별 가능한 최소 효과 = {SCREEN_RESOLVABLE:+.3f} (fold 차분 표준편차 {SCREEN_SD})")


스크리닝 fold = 0 (고정)
판별 가능한 최소 효과 = +0.040 (fold 차분 표준편차 0.02)


### 기준선 불러오기 — 학습하지 않는다

`GIN[base]` fold 0 은 이미 학습돼 캐시에 있다. `(캐시)` 표시가 뜨면 재사용된 것이다.


In [ ]:
BASE_CFG = dict(conv="gin", balanced_sampler=False, use_class_weights=True, jk=None)
base = screen("GIN[base]", BASE_CFG, cache_dir=CACHE_DIR)   # 기존 캐시 위치

print(f"\n기준선 (fold {SCREEN_FOLD}) macro F1 = {base['test_macro_f1']:.4f}")
print(f"참고: 5 fold 평균 0.8445 ± 0.0148 / 봉인 최종 0.8662\n")
print(per_class_with_support(base, names).to_string())


장치: cuda | 설정 ['GIN[base]'] | fold [0] | epoch 50

train 14296 / val 3568 / eval 3568  (증강: True, 균형샘플러: False)

--- fold 0 / GIN[base] ---
  [GIN         ] Epoch   1/50 | Train 1.7851 | Val 1.5054 | Val F1 0.3820 *
  [GIN         ] Epoch   2/50 | Train 1.1941 | Val 0.9119 | Val F1 0.5703 *
  [GIN         ] Epoch   3/50 | Train 0.8389 | Val 0.6659 | Val F1 0.6613 *
  [GIN         ] Epoch   4/50 | Train 0.6663 | Val 0.6166 | Val F1 0.6929 *
  [GIN         ] Epoch   5/50 | Train 0.5899 | Val 0.4962 | Val F1 0.7343 *
  [GIN         ] Epoch   7/50 | Train 0.5004 | Val 0.4006 | Val F1 0.7588 *
  [GIN         ] Epoch   9/50 | Train 0.4541 | Val 0.3752 | Val F1 0.7820 *
  [GIN         ] Epoch  10/50 | Train 0.4428 | Val 0.4605 | Val F1 0.7071
  [GIN         ] Epoch  11/50 | Train 0.4088 | Val 0.3693 | Val F1 0.7865 *
  [GIN         ] Epoch  14/50 | Train 0.4041 | Val 0.3336 | Val F1 0.8173 *
  [GIN         ] Epoch  20/50 | Train 0.3507 | Val 0.4616 | Val F1 0.7579
  [GIN         ] Epoch  21/5

# 11. ① 극좌표 CNN 단독 — 융합 진행 여부를 가르는 단계

**CNN 이 GIN 보다 잘하는지는 중요하지 않다.** 중요한 것은 **서로 다른 웨이퍼에서
틀리는가**다. 같은 곳에서 같이 틀리면 합쳐도 소용이 없다.

약 25분.


In [ ]:
CNN_CFG = dict(conv="gin", balanced_sampler=False, use_class_weights=True, jk=None,
               build=lambda: PolarCNNClassifier(hidden=HIDDEN, width=32))
cnn = screen("PolarCNN", CNN_CFG)

print(f"\nCNN 단독  macro F1 {cnn['test_macro_f1']:.4f}   (기준선 {base['test_macro_f1']:.4f})")
print()
print(per_class_with_support(cnn, names).to_string())


장치: cuda | 설정 ['PolarCNN'] | fold [0] | epoch 50

train 14296 / val 3568 / eval 3568  (증강: True, 균형샘플러: False)

--- fold 0 / PolarCNN ---
  [GIN         ] Epoch   1/50 | Train 1.5252 | Val 1.0192 | Val F1 0.4905 *
  [GIN         ] Epoch   2/50 | Train 0.8539 | Val 0.6230 | Val F1 0.6464 *
  [GIN         ] Epoch   3/50 | Train 0.5732 | Val 0.5661 | Val F1 0.6773 *
  [GIN         ] Epoch   4/50 | Train 0.4651 | Val 0.3371 | Val F1 0.8114 *
  [GIN         ] Epoch   6/50 | Train 0.3710 | Val 0.3089 | Val F1 0.8145 *
  [GIN         ] Epoch   7/50 | Train 0.3561 | Val 0.2898 | Val F1 0.8278 *
  [GIN         ] Epoch   9/50 | Train 0.3014 | Val 0.2566 | Val F1 0.8336 *
  [GIN         ] Epoch  10/50 | Train 0.2798 | Val 0.2605 | Val F1 0.8253
  [GIN         ] Epoch  11/50 | Train 0.2841 | Val 0.2459 | Val F1 0.8552 *
  [GIN         ] Epoch  12/50 | Train 0.2523 | Val 0.2221 | Val F1 0.8572 *
  [GIN         ] Epoch  16/50 | Train 0.2328 | Val 0.2253 | Val F1 0.8931 *
  [GIN         ] Epoch  20/5

### 오라클 진단 — 융합의 천장

웨이퍼마다 둘 중 맞은 쪽을 골라주는 가상의 완벽한 심판이다.
**어떤 융합도 이 값을 넘을 수 없다.**

| 여지 (오라클 − 최고 단일) | 결정 |
|---|---|
| **+0.05 이상** | 융합 진행 |
| +0.02 ~ +0.05 | 애매. 시간 여유가 있으면 |
| **+0.02 미만** | **융합 포기.** 천장이 낮아 어떻게 만들어도 안 됨 |

참고: 이전에 GIN+GAT 오라클은 여지가 +0.01 수준이었고 실제 앙상블 이득도
+0.021 에 그쳤다. 두 그래프 모델이 너무 비슷했기 때문이다.


In [ ]:
orc = oracle_upper_bound(base, cnn, class_names=names)

print("=" * 62)
print(f"{orc['a']}  {orc['macro_a']:.4f}")
print(f"{orc['b']}  {orc['macro_b']:.4f}")
print(f"오라클 상한   {orc['oracle']:.4f}")
print(f"여지          {orc['여지']:+.4f}")
print("=" * 62)
print(f"판정: {orc['판정']}\n")

print(f"둘 다 맞음 {orc['둘다맞음']:.3f} | GIN만 {orc['A만맞음']:.3f} | "
      f"CNN만 {orc['B만맞음']:.3f} | 둘 다 틀림 {orc['둘다틀림']:.3f}")
print("\n'CNN만 맞음' 이 클수록 융합 여지가 크다. 0 에 가까우면 CNN 이 기여할 게 없다.\n")

d = pd.DataFrame(orc["클래스별"]).set_index("패턴")
print(d.round(4).to_string())
print("\n특히 Scratch 를 보라 — 극좌표에서 직선이 곡선이 되므로 CNN 이 불리할 수 있다.")

GO_FUSION = orc["여지"] >= 0.02
print(f"\n=> 융합 진행 여부: {'진행' if GO_FUSION else '포기'}")


GIN[base]  0.8314
PolarCNN  0.9131
오라클 상한   0.9521
여지          +0.0390
판정: 애매 — 시간 여유가 있으면

둘 다 맞음 0.874 | GIN만 0.030 | CNN만 0.059 | 둘 다 틀림 0.037

'CNN만 맞음' 이 클수록 융합 여지가 크다. 0 에 가까우면 CNN 이 기여할 게 없다.

            지원수    A정답률    B정답률    B만맞음
패턴                                     
Center      607  0.9555  0.9588  0.0247
Donut        80  0.9375  0.9500  0.0500
Edge-Loc    721  0.8738  0.8988  0.0763
Edge-Ring  1351  0.9534  0.9830  0.0333
Loc         500  0.7720  0.8480  0.1360
Random      123  0.8699  0.9187  0.0976
Scratch     163  0.8405  0.8221  0.0675
Near-full    23  1.0000  1.0000  0.0000

특히 Scratch 를 보라 — 극좌표에서 직선이 곡선이 되므로 CNN 이 불리할 수 있다.

=> 융합 진행 여부: 진행


# 12. ② 클래스 가중치 완화

봉인 평가에서 드러난 **과교정**을 고친다.

```
macro 재현율 0.914  vs  정밀도 0.833     (0.08 차이)
Donut     P 0.708 / R 0.960
Near-full P 0.714 / R 1.000
Loc       P 0.871 / R 0.743   <- 274장을 잃었다
```

가중치 `w = (N / (C·n_c))^alpha` 에서 `alpha` 를 낮춘다.

| alpha | Edge-Ring : Near-full 벌점비 |
|---|---|
| 1.0 (현재) | 65 : 1 |
| **0.5** | **8 : 1** |
| 0.25 | 2.8 : 1 |

F1 은 정밀도와 재현율이 비슷할 때 최대이므로, 한쪽으로 치우친 현재 지점은
소수 클래스와 Loc **양쪽 모두** 손해다.

약 20분.


In [ ]:
A05_CFG = dict(conv="gin", balanced_sampler=False, use_class_weights=True, jk=None,
               weight_alpha=0.5)
a05 = screen("GIN[a=0.5]", A05_CFG)

delta = a05["test_macro_f1"] - base["test_macro_f1"]
verdict, action = screen_verdict(delta)
print(f"\n기준선   {base['test_macro_f1']:.4f}")
print(f"alpha=0.5 {a05['test_macro_f1']:.4f}   차이 {delta:+.4f}")
print(f"판정: {verdict} — {action}\n")

cmp = pd.DataFrame({
    "base_F1": base["per_class_f1"], "a05_F1": a05["per_class_f1"],
}, index=names)
cmp["차이"] = cmp["a05_F1"] - cmp["base_F1"]
print(cmp.round(4).to_string())

from sklearn.metrics import precision_score, recall_score
for nm, r in [("base", base), ("a=0.5", a05)]:
    p = precision_score(r["labels"], r["preds"], average="macro", zero_division=0)
    rc = recall_score(r["labels"], r["preds"], average="macro", zero_division=0)
    print(f"\n{nm:<6} macro 정밀도 {p:.4f} / 재현율 {rc:.4f}  (간격 {rc-p:+.4f})")
print("\n간격이 좁아졌으면 과교정이 완화된 것이다.")


장치: cuda | 설정 ['GIN[a=0.5]'] | fold [0] | epoch 50

train 14296 / val 3568 / eval 3568  (증강: True, 균형샘플러: False)

--- fold 0 / GIN[a=0.5] ---
  [GIN         ] Epoch   1/50 | Train 1.6423 | Val 1.0220 | Val F1 0.5932 *
  [GIN         ] Epoch   2/50 | Train 0.9023 | Val 0.6836 | Val F1 0.6633 *
  [GIN         ] Epoch   3/50 | Train 0.6573 | Val 0.4731 | Val F1 0.7661 *
  [GIN         ] Epoch   6/50 | Train 0.4633 | Val 0.3893 | Val F1 0.8017 *
  [GIN         ] Epoch   7/50 | Train 0.4314 | Val 0.3741 | Val F1 0.8170 *
  [GIN         ] Epoch   9/50 | Train 0.3886 | Val 0.3134 | Val F1 0.8421 *
  [GIN         ] Epoch  10/50 | Train 0.3846 | Val 0.3243 | Val F1 0.8440 *
  [GIN         ] Epoch  14/50 | Train 0.3478 | Val 0.2843 | Val F1 0.8684 *
  [GIN         ] Epoch  20/50 | Train 0.3126 | Val 0.2910 | Val F1 0.8583
  [GIN         ] Epoch  21/50 | Train 0.3135 | Val 0.2622 | Val F1 0.8723 *
  [GIN         ] Epoch  30/50 | Train 0.2844 | Val 0.2525 | Val F1 0.8672
  [GIN         ] 10 epoch 

# 13. ③ 게이트 융합 — ①이 통과했을 때만

`GO_FUSION` 이 False 면 이 셀을 건너뛴다. 오라클 천장이 낮으면 어떤 융합을
만들어도 그 위로 못 간다.

게이트는 웨이퍼마다 "그래프를 얼마나 볼지"를 **차원별로** 학습한다.
게이트 평균이 0 이나 1 에 붙으면 한쪽 트랙을 버린 것이므로 반드시 확인한다.

약 35분.


In [ ]:
if not GO_FUSION:
    print("① 오라클 여지가 부족해 융합을 건너뛴다.")
    print(f"   여지 {orc['여지']:+.4f} < +0.02")
    print("   -> ②(가중치 완화) 결과만 가지고 ④단계로 간다.")
else:
    FUSE_CFG = dict(conv="gin", balanced_sampler=False, use_class_weights=True, jk=None,
                    build=lambda: GatedFusion(hidden=HIDDEN, conv="gin",
                                              num_layers=NUM_LAYERS, heads=HEADS,
                                              width=32))
    fuse = screen("GatedFusion", FUSE_CFG)
    d = fuse["test_macro_f1"] - base["test_macro_f1"]
    v, act = screen_verdict(d)
    print(f"\n기준선 {base['test_macro_f1']:.4f} / 융합 {fuse['test_macro_f1']:.4f}"
          f"   차이 {d:+.4f}")
    print(f"판정: {v} — {act}\n")
    print(pd.DataFrame({"base": base["per_class_f1"], "fusion": fuse["per_class_f1"],
                        "차이": fuse["per_class_f1"] - base["per_class_f1"]},
                       index=names).round(4).to_string())


장치: cuda | 설정 ['GatedFusion'] | fold [0] | epoch 50

train 14296 / val 3568 / eval 3568  (증강: True, 균형샘플러: False)

--- fold 0 / GatedFusion ---
  [GIN         ] Epoch   1/50 | Train 1.7496 | Val 1.3221 | Val F1 0.4716 *
  [GIN         ] Epoch   2/50 | Train 1.0108 | Val 0.6701 | Val F1 0.6859 *
  [GIN         ] Epoch   3/50 | Train 0.6340 | Val 0.5279 | Val F1 0.7375 *
  [GIN         ] Epoch   4/50 | Train 0.5075 | Val 0.4011 | Val F1 0.8027 *
  [GIN         ] Epoch   5/50 | Train 0.4379 | Val 0.3164 | Val F1 0.8041 *
  [GIN         ] Epoch   6/50 | Train 0.4095 | Val 0.2699 | Val F1 0.8394 *
  [GIN         ] Epoch   9/50 | Train 0.3372 | Val 0.2607 | Val F1 0.8562 *
  [GIN         ] Epoch  10/50 | Train 0.3397 | Val 0.2712 | Val F1 0.8152
  [GIN         ] Epoch  13/50 | Train 0.2784 | Val 0.2100 | Val F1 0.8646 *
  [GIN         ] Epoch  14/50 | Train 0.2813 | Val 0.2066 | Val F1 0.8974 *
  [GIN         ] Epoch  20/50 | Train 0.2206 | Val 0.1903 | Val F1 0.8864
  [GIN         ] 10 epoc

# 14. 스크리닝 종합 — 무엇을 5 fold 로 올릴 것인가

`유망` 이상만 ④단계로 보낸다. `판정 불가` 는 1 fold 해상도 밖이므로,
포기하거나 5 fold 로 확인해야 한다.


In [ ]:
rows = []
for nm, r in SCREEN.items():
    if nm == "GIN[base]":
        continue
    d = r["test_macro_f1"] - base["test_macro_f1"]
    v, act = screen_verdict(d)
    rows.append({"설정": nm, "fold0_F1": r["test_macro_f1"], "기준선대비": d,
                 "판정": v, "다음": act})

df = pd.DataFrame(rows).sort_values("기준선대비", ascending=False)
print("=" * 92)
print(f"1 fold 스크리닝 종합  (기준선 GIN[base] fold{SCREEN_FOLD} = {base['test_macro_f1']:.4f})")
print("=" * 92)
print(df.round(4).to_string(index=False))

promote = df[df["판정"].isin(["채택 유력", "유망"])]["설정"].tolist()
print(f"\n5 fold 로 올릴 후보: {promote if promote else '없음'}")
if not promote:
    print("  전부 판별 해상도 밖이다. 두 선택지가 있다:")
    print("   - 여기서 멈추고 0.8662 로 확정한다")
    print("   - 가장 유망한 하나를 5 fold 로 확인한다 (1.5시간)")

print(f"\n[참고] 1 fold 판별 해상도: |차이| > {2*SCREEN_SD:.3f} 이어야 신뢰할 수 있다.")


1 fold 스크리닝 종합  (기준선 GIN[base] fold0 = 0.8314)
         설정  fold0_F1  기준선대비    판정                 다음
   PolarCNN    0.9131 0.0817 채택 유력 5 fold 는 최종 후보일 때만
GatedFusion    0.8974 0.0660 채택 유력 5 fold 는 최종 후보일 때만
 GIN[a=0.5]    0.8723 0.0408    유망     5 fold 로 확정할 것

5 fold 로 올릴 후보: ['PolarCNN', 'GatedFusion', 'GIN[a=0.5]']

[참고] 1 fold 판별 해상도: |차이| > 0.040 이어야 신뢰할 수 있다.


# 15. ④ 5 fold 확정 — 후보가 있을 때만

`PROMOTE` 에 올릴 설정 이름을 넣고 실행한다. 설정당 약 1.5시간.

여기서 나온 대응 비교가 **논문에 쓸 근거**다. 1 fold 결과는 탐색용이지
보고용이 아니다.


In [ ]:
PROMOTE = promote        # 필요하면 직접 지정: ["GIN[a=0.5]"]

CFG_BY_NAME = {"GIN[a=0.5]": A05_CFG, "PolarCNN": CNN_CFG}
if GO_FUSION:
    CFG_BY_NAME["GatedFusion"] = FUSE_CFG

FULL = []
if not PROMOTE:
    print("올릴 후보가 없다. 이 셀을 건너뛴다.")
else:
    # 기준선의 나머지 fold 는 기존 캐시에서 즉시 로드된다.
    FULL += run_dev_stage(packed, split, {"GIN[base]": BASE_CFG}, PREPROC_HASH,
                          split_mode=split_meta["mode"], seed=SEED, epochs=EPOCHS,
                          batch_size=BATCH_SIZE, lr=LR, weight_decay=WD,
                          patience=PATIENCE, hidden=HIDDEN, num_layers=NUM_LAYERS,
                          heads=HEADS, num_workers=2, cache_dir=CACHE_DIR,
                          ckpt_dir=CKPT_DIR)
    for nm in PROMOTE:
        FULL += run_dev_stage(packed, split, {nm: CFG_BY_NAME[nm]}, PREPROC_HASH,
                              split_mode=split_meta["mode"], seed=SEED, epochs=EPOCHS,
                              batch_size=BATCH_SIZE, lr=LR, weight_decay=WD,
                              patience=PATIENCE, hidden=HIDDEN, num_layers=NUM_LAYERS,
                              heads=HEADS, num_workers=2, cache_dir=EXT_CACHE,
                              ckpt_dir=EXT_CKPT)
    print("\n" + "=" * 64)
    print("§5.4 개발 지표")
    print("=" * 64)
    print(dev_summary(FULL).round(4).to_string())
    print(f"\n각주: {FOLD_SD_FOOTNOTE}")

    print("\n\n§5.2 대응 비교 — GIN[base] 기준")
    print("=" * 64)
    print(paired_table(FULL, "GIN[base]").round(4).to_string(index=False))
    for nm in PROMOTE:
        c = paired_comparison(FULL, "GIN[base]", nm)
        print(f"\n  {nm}: " + "  ".join(f"{d:+.4f}" for d in c["fold별차"]))


장치: cuda | 설정 ['GIN[base]'] | fold [0, 1, 2, 3, 4] | epoch 50

train 14296 / val 3568 / eval 3568  (증강: True, 균형샘플러: False)

--- fold 0 / GIN[base] ---
  (캐시) 검증 macro F1 0.8314
train 14288 / val 3576 / eval 3576  (증강: True, 균형샘플러: False)

--- fold 1 / GIN[base] ---
  (캐시) 검증 macro F1 0.8205
train 14288 / val 3576 / eval 3576  (증강: True, 균형샘플러: False)

--- fold 2 / GIN[base] ---
  (캐시) 검증 macro F1 0.8450
train 14290 / val 3574 / eval 3574  (증강: True, 균형샘플러: False)

--- fold 3 / GIN[base] ---
  (캐시) 검증 macro F1 0.8600
train 14294 / val 3570 / eval 3570  (증강: True, 균형샘플러: False)

--- fold 4 / GIN[base] ---
  (캐시) 검증 macro F1 0.8521
장치: cuda | 설정 ['PolarCNN'] | fold [0, 1, 2, 3, 4] | epoch 50

train 14296 / val 3568 / eval 3568  (증강: True, 균형샘플러: False)

--- fold 0 / PolarCNN ---
  (캐시) 검증 macro F1 0.9131
train 14288 / val 3576 / eval 3576  (증강: True, 균형샘플러: False)

--- fold 1 / PolarCNN ---
  [GIN         ] Epoch   1/50 | Train 1.5690 | Val 1.1168 | Val F1 0.5426 *
  [GIN         ] Epoch

# 16. 최종 판단

**5 fold 대응 비교에서 개선 4/5 fold 이상**이면 새 구성을 최종 후보로 올린다.
그 경우에만 봉인 집합을 한 번 더 연다 — 그리고 **결과가 나쁘게 나와도 그대로 보고한다.**

3/5 이하면 "유의한 차이를 관측하지 못했다" 로 보고하고 `0.8662` 를 유지한다.

봉인 재개봉이 필요하면 `ACK2026_Wafer_Spec_Colab.ipynb` 의 셀 21~22 를
새 구성으로 실행한다. 이 노트북은 봉인을 열지 않는다.


In [ ]:
if not PROMOTE or not FULL:
    print("5 fold 결과가 없다. 현재 확정값을 유지한다.")
    print("\n  최종: GIN[base]  macro F1 0.8662  (봉인 7,655장)")
else:
    print("=" * 70)
    print("최종 판단")
    print("=" * 70)
    for nm in PROMOTE:
        c = paired_comparison(FULL, "GIN[base]", nm)
        ok = c["개선fold"] >= 4
        print(f"\n{nm}")
        print(f"  {c['표기']}")
        print(f"  -> {'봉인 재개봉 대상' if ok else '유의한 차이 없음. 기존 값 유지'}")

    print("\n" + "-" * 70)
    print("어느 쪽이든 이미 확정된 0.8662 는 논문에 그대로 남는다.")
    print("새 구성을 채택하면 봉인 결과를 **추가로** 보고하되,")
    print("'최종 평가 집합은 모델별로 1회씩 사용되었고 그 결과에 따른")
    print(" 구성 변경은 없었다' 를 명시한다.")


최종 판단

PolarCNN
  +0.0642 ± 0.0178 (개선 5/5 fold)
  -> 봉인 재개봉 대상

GatedFusion
  +0.0566 ± 0.0210 (개선 5/5 fold)
  -> 봉인 재개봉 대상

GIN[a=0.5]
  +0.0179 ± 0.0269 (개선 3/5 fold)
  -> 유의한 차이 없음. 기존 값 유지

----------------------------------------------------------------------
어느 쪽이든 이미 확정된 0.8662 는 논문에 그대로 남는다.
새 구성을 채택하면 봉인 결과를 **추가로** 보고하되,
'최종 평가 집합은 모델별로 1회씩 사용되었고 그 결과에 따른
 구성 변경은 없었다' 를 명시한다.
